# Wire line S3 Second connector: auto-label, review, train, export

Free Colab GPU, no API. **First:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells
top to bottom (Runtime → Run all).

1. Upload `frames.zip` from `training/s3/` (or the loose frames).
2. Grounding DINO draws boxes for this station's classes on every frame.
3. **Review every frame** (the team guide asks for every box to be checked) and drop the bad ones.
4. Train a small YOLO model (10–20 min).
5. Download **`s3-connector-v3.onnx`** and send it to Claude (or put it in `app/models/`).

Classes, in this exact order (the order `/line` expects): `red_wire, black_wire, connector_open, connector_closed`

**Nov 19 update:** adds 131 frames from the live S3 camera (yellow table, sessions `s3-16`–`s3-18`), 7 phone photos (`s3-p1`) and 19 frames with nothing on them (empty table, phone face down), which teach the model that a dark picture is not a black wire. Live session `s3-17` is in the check set, so the score says how the model does on the real camera. Cell 7b scores the current model (`s3-connector-new.onnx`) on the same check set.

In [ ]:
# 1. GPU check and install
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU! Runtime > Change runtime type > T4 GPU"
!pip -q install -U ultralytics onnx
import torch; print("GPU available:", torch.cuda.is_available())

In [ ]:
# 2. Upload frames.zip (training/s3/frames.zip on your laptop). Loose .jpg frames also work.
import os, zipfile, glob, shutil
from pathlib import Path

WORK = Path("/content/s3")
(WORK / "frames").mkdir(parents=True, exist_ok=True)
if not glob.glob(str(WORK / "frames" / "*.jpg")):
    from google.colab import files
    print("Choose training/s3/frames.zip ...")
    for name in files.upload():
        if zipfile.is_zipfile(name):
            zipfile.ZipFile(name).extractall(WORK)
        elif name.lower().endswith((".jpg", ".jpeg", ".png")):
            shutil.move(name, WORK / "frames" / name)
FRAMES = sorted(glob.glob(str(WORK / "frames" / "*.jpg")))
VIDEOS = sorted({Path(f).stem.rsplit("_", 1)[0] for f in FRAMES})
print(len(FRAMES), "frames from", len(VIDEOS), "videos:", VIDEOS)

In [ ]:
# 3a. Settings
STATION = "s3"
CLASSES = ["red_wire", "black_wire", "connector_open", "connector_closed"]          # do not reorder: /line reads them in this order
PROMPT = "red wire. black wire. orange lever connector."
ONE_PER_IMAGE = []   # classes that exist once per frame
BOX_THRESHOLD, TEXT_THRESHOLD = 0.30, 0.25
DINO_MODEL = "IDEA-Research/grounding-dino-base"
EXPORT_NAME = "s3-connector-v3.onnx"
FPS = 2                          # frames per second pulled from each video
VAL_VIDEOS = ["s3-04", "s3-09", "s3-13", "s3-17"]   # whole videos kept out of training to check the model

### 3a-2. Connector boxes (already labelled)

The connector boxes and open/closed come from Claude: **1303 frames** where it is clear which
connector is the arrived one (always closed) and which is the new one, plus the S2 frames (`s2-...`: the same lever
connector, many tables and lights) so the model sees enough open connectors. Grounding DINO only adds the wires.
The check set is S3 videos only.
Frames not in this list are left out. In the review, **green** = open, **purple** = closed.

**Nov 19:** plus the live-camera frames (`s3-16`–`s3-18`) and phone photos (`s3-p1`), with connector boxes from Claude: the arrived connector is closed, the new one is open while it is in the hand being filled, and both are closed once the unit is done. Frames in `EMPTY` get no boxes at all (not even wires).

In [ ]:
CONN = {"s3-01_0001": [["connector_closed", 0.6471, 0.5183, 0.713, 0.5577]], "s3-01_0002": [["connector_closed", 0.6375, 0.5226, 0.7052, 0.5615]], "s3-01_0003": [["connector_closed", 0.6597, 0.5294, 0.7283, 0.5701]], "s3-01_0004": [["connector_closed", 0.6824, 0.5407, 0.7521, 0.5814]], "s3-01_0005": [["connector_closed", 0.7124, 0.5244, 0.7767, 0.5619]], "s3-01_0006": [["connector_closed", 0.7245, 0.5166, 0.7853, 0.5555]], "s3-01_0007": [["connector_closed", 0.1807, 0.4524, 0.2454, 0.4891]], "s3-01_0008": [["connector_closed", 0.7185, 0.5156, 0.782, 0.5528], ["connector_closed", 0.1774, 0.4178, 0.2426, 0.458]], "s3-01_0009": [["connector_closed", 0.1721, 0.4417, 0.2367, 0.4821]], "s3-01_0010": [["connector_closed", 0.1557, 0.4605, 0.2186, 0.4989]], "s3-01_0020": [["connector_closed", 0.1182, 0.4138, 0.1851, 0.4504], ["connector_open", 0.6782, 0.5013, 0.7269, 0.5383]], "s3-01_0023": [["connector_closed", 0.1242, 0.3967, 0.1919, 0.434], ["connector_open", 0.6342, 0.4964, 0.7126, 0.5374]], "s3-01_0024": [["connector_closed", 0.1126, 0.3976, 0.1836, 0.4357], ["connector_open", 0.6329, 0.5093, 0.7041, 0.548]], "s3-01_0025": [["connector_closed", 0.1124, 0.3972, 0.1849, 0.4368], ["connector_open", 0.6326, 0.5128, 0.703, 0.5495]], "s3-01_0026": [["connector_open", 0.6324, 0.5115, 0.7013, 0.5492], ["connector_closed", 0.109, 0.3961, 0.1825, 0.4345]], "s3-01_0027": [["connector_closed", 0.0893, 0.3934, 0.1679, 0.4318], ["connector_open", 0.593, 0.519, 0.6653, 0.5639]], "s3-01_0028": [["connector_closed", 0.074, 0.4212, 0.1451, 0.4606], ["connector_open", 0.59, 0.5273, 0.6644, 0.5707]], "s3-01_0029": [["connector_closed", 0.1032, 0.3927, 0.1688, 0.4265], ["connector_open", 0.6177, 0.5078, 0.6827, 0.5473]], "s3-01_0030": [["connector_open", 0.6293, 0.4562, 0.7002, 0.5046], ["connector_closed", 0.0969, 0.371, 0.1613, 0.4008]], "s3-01_0031": [["connector_open", 0.6353, 0.472, 0.7017, 0.5074]], "s3-01_0032": [["connector_open", 0.6341, 0.4667, 0.6988, 0.5066]], "s3-01_0033": [["connector_open", 0.6381, 0.4486, 0.701, 0.4854]], "s3-01_0034": [["connector_open", 0.6347, 0.433, 0.701, 0.4741], ["connector_closed", 0.0799, 0.3912, 0.1438, 0.428]], "s3-01_0035": [["connector_open", 0.636, 0.4326, 0.6994, 0.4725], ["connector_closed", 0.0816, 0.3898, 0.1466, 0.4271]], "s3-01_0036": [["connector_closed", 0.0834, 0.3888, 0.1478, 0.4239], ["connector_open", 0.6404, 0.4333, 0.6998, 0.4709]], "s3-01_0040": [["connector_open", 0.671, 0.4451, 0.7366, 0.4823], ["connector_closed", 0.117, 0.3879, 0.1804, 0.4229]], "s3-01_0041": [["connector_open", 0.6672, 0.4415, 0.7313, 0.4807], ["connector_closed", 0.1122, 0.386, 0.1748, 0.4191]], "s3-01_0042": [["connector_closed", 0.1088, 0.3833, 0.1722, 0.4159], ["connector_open", 0.6625, 0.4442, 0.7273, 0.4801]], "s3-01_0043": [["connector_closed", 0.1085, 0.3822, 0.17, 0.4136], ["connector_open", 0.6635, 0.4456, 0.7237, 0.4809]], "s3-01_0044": [["connector_closed", 0.1084, 0.3783, 0.1734, 0.4152], ["connector_open", 0.6648, 0.4476, 0.7247, 0.4862]], "s3-01_0045": [["connector_closed", 0.1119, 0.3747, 0.1754, 0.4091], ["connector_open", 0.6651, 0.4463, 0.7323, 0.487]], "s3-01_0056": [["connector_closed", 0.6582, 0.4501, 0.7239, 0.4913], ["connector_closed", 0.099, 0.3714, 0.1679, 0.4088]], "s3-01_0057": [["connector_closed", 0.7045, 0.4703, 0.7727, 0.5114], ["connector_closed", 0.1168, 0.3883, 0.1861, 0.4303]], "s3-01_0058": [["connector_closed", 0.7219, 0.4806, 0.795, 0.523], ["connector_closed", 0.1065, 0.3994, 0.1785, 0.439]], "s3-01_0059": [["connector_closed", 0.7192, 0.4872, 0.7916, 0.5299], ["connector_closed", 0.0943, 0.4093, 0.1683, 0.4498]], "s3-01_0060": [["connector_closed", 0.7231, 0.4902, 0.7995, 0.5345], ["connector_closed", 0.0923, 0.4156, 0.1682, 0.4603]], "s3-02_0001": [["connector_closed", 0.5109, 0.4315, 0.5809, 0.4712]], "s3-02_0002": [["connector_closed", 0.4652, 0.4658, 0.538, 0.5082], ["connector_closed", 0.1134, 0.7052, 0.1747, 0.7487]], "s3-02_0003": [["connector_closed", 0.1074, 0.7429, 0.1833, 0.788], ["connector_closed", 0.4729, 0.5335, 0.5423, 0.5748]], "s3-02_0004": [["connector_closed", 0.1379, 0.7412, 0.2146, 0.7868], ["connector_closed", 0.4916, 0.5367, 0.5602, 0.5782]], "s3-02_0005": [["connector_closed", 0.5294, 0.5293, 0.5994, 0.5696], ["connector_closed", 0.1931, 0.65, 0.2649, 0.6841]], "s3-02_0017": [["connector_closed", 0.1269, 0.3439, 0.2071, 0.3821], ["connector_open", 0.6361, 0.4979, 0.6988, 0.535]], "s3-02_0018": [["connector_closed", 0.1697, 0.3331, 0.2493, 0.3747], ["connector_open", 0.6412, 0.5006, 0.71, 0.5379]], "s3-02_0019": [["connector_closed", 0.1781, 0.3301, 0.256, 0.3712], ["connector_open", 0.6369, 0.4971, 0.7032, 0.5364]], "s3-02_0021": [["connector_closed", 0.167, 0.3227, 0.2455, 0.3629], ["connector_open", 0.6307, 0.4923, 0.6919, 0.5322]], "s3-02_0022": [["connector_closed", 0.1653, 0.3184, 0.2438, 0.3589], ["connector_open", 0.561, 0.5215, 0.6312, 0.5629]], "s3-02_0023": [["connector_open", 0.6145, 0.4849, 0.6761, 0.524]], "s3-02_0024": [["connector_open", 0.6211, 0.4742, 0.6902, 0.5166]], "s3-02_0025": [["connector_open", 0.6223, 0.4786, 0.6927, 0.5174]], "s3-02_0026": [["connector_open", 0.619, 0.4778, 0.6959, 0.5209], ["connector_closed", 0.0691, 0.3989, 0.1367, 0.4351]], "s3-02_0027": [["connector_open", 0.61, 0.4745, 0.6847, 0.5169], ["connector_closed", 0.0643, 0.3948, 0.1327, 0.4306]], "s3-02_0028": [["connector_open", 0.6228, 0.4719, 0.698, 0.5169], ["connector_closed", 0.0742, 0.3916, 0.1425, 0.43]], "s3-02_0029": [["connector_open", 0.6274, 0.4724, 0.7023, 0.5161], ["connector_closed", 0.0796, 0.3925, 0.149, 0.4287]], "s3-02_0034": [["connector_closed", 0.1286, 0.3793, 0.1991, 0.4138], ["connector_open", 0.6472, 0.5242, 0.7144, 0.5618]], "s3-02_0035": [["connector_closed", 0.1312, 0.3671, 0.2069, 0.4068], ["connector_open", 0.6414, 0.5239, 0.7109, 0.5643]], "s3-02_0036": [["connector_closed", 0.1411, 0.3643, 0.2189, 0.4016], ["connector_open", 0.6491, 0.5267, 0.7122, 0.5664]], "s3-02_0037": [["connector_closed", 0.1445, 0.3582, 0.2203, 0.3959], ["connector_open", 0.6357, 0.5284, 0.7066, 0.5676]], "s3-02_0038": [["connector_closed", 0.1423, 0.3505, 0.2234, 0.3914], ["connector_open", 0.6327, 0.5217, 0.7037, 0.5603]], "s3-02_0041": [["connector_closed", 0.1571, 0.3156, 0.2386, 0.356], ["connector_open", 0.6076, 0.5207, 0.6767, 0.5575]], "s3-02_0042": [["connector_closed", 0.163, 0.3143, 0.2479, 0.3553], ["connector_open", 0.6315, 0.5111, 0.711, 0.5473]], "s3-02_0043": [["connector_closed", 0.149, 0.3015, 0.2368, 0.3476], ["connector_open", 0.588, 0.5151, 0.6793, 0.5588]], "s3-02_0044": [["connector_closed", 0.1764, 0.3089, 0.2631, 0.3549], ["connector_open", 0.5816, 0.5359, 0.665, 0.5801]], "s3-02_0045": [["connector_closed", 0.1159, 0.32, 0.184, 0.3559], ["connector_open", 0.6481, 0.4637, 0.7176, 0.5056]], "s3-02_0046": [["connector_open", 0.6507, 0.4758, 0.7201, 0.5131]], "s3-02_0050": [["connector_closed", 0.7, 0.4905, 0.7795, 0.5348], ["connector_closed", 0.1489, 0.3938, 0.2184, 0.4353]], "s3-02_0051": [["connector_closed", 0.1517, 0.407, 0.221, 0.4477]], "s3-02_0052": [["connector_closed", 0.1428, 0.4158, 0.2126, 0.4589]], "s3-02_0053": [["connector_closed", 0.1434, 0.4204, 0.2144, 0.461]], "s3-03_0007": [["connector_closed", 0.1021, 0.7069, 0.2012, 0.7626]], "s3-03_0019": [["connector_open", 0.5391, 0.1839, 0.6401, 0.243]], "s3-03_0020": [["connector_closed", 0.2045, 0.5732, 0.3187, 0.6338], ["connector_open", 0.5313, 0.1822, 0.6339, 0.2405]], "s3-03_0029": [["connector_closed", 0.5548, 0.1831, 0.6574, 0.2395]], "s3-03_0030": [["connector_closed", 0.1715, 0.5603, 0.2805, 0.6215], ["connector_closed", 0.5488, 0.1832, 0.6515, 0.2397]], "s3-03_0031": [["connector_closed", 0.1669, 0.5653, 0.281, 0.6245], ["connector_closed", 0.5424, 0.1832, 0.6467, 0.2401]], "s3-04_0001": [["connector_closed", 0.6049, 0.7154, 0.6834, 0.7633]], "s3-04_0002": [["connector_closed", 0.6009, 0.716, 0.6817, 0.7648]], "s3-04_0003": [["connector_closed", 0.6055, 0.714, 0.6828, 0.762]], "s3-04_0004": [["connector_closed", 0.5996, 0.7076, 0.6736, 0.7465]], "s3-04_0034": [["connector_closed", 0.451, 0.2234, 0.5256, 0.2649], ["connector_closed", 0.5286, 0.4936, 0.6065, 0.5371]], "s3-04_0035": [["connector_closed", 0.4431, 0.2283, 0.5234, 0.2712], ["connector_closed", 0.5279, 0.4963, 0.6022, 0.5388]], "s3-04_0036": [["connector_closed", 0.4385, 0.2317, 0.5203, 0.2739], ["connector_closed", 0.5224, 0.4996, 0.5962, 0.5443]], "s3-05_0001": [["connector_closed", 0.607, 0.394, 0.65, 0.4726]], "s3-05_0002": [["connector_closed", 0.6007, 0.3919, 0.6459, 0.4712]], "s3-05_0003": [["connector_closed", 0.5999, 0.3955, 0.6461, 0.4769]], "s3-05_0011": [["connector_open", 0.4999, 0.4344, 0.5393, 0.5012]], "s3-05_0012": [["connector_open", 0.498, 0.4199, 0.5346, 0.4876]], "s3-05_0013": [["connector_open", 0.4961, 0.416, 0.5322, 0.4846]], "s3-05_0014": [["connector_open", 0.4719, 0.4166, 0.5172, 0.4898]], "s3-05_0015": [["connector_open", 0.4757, 0.4239, 0.5098, 0.4835]], "s3-05_0024": [["connector_closed", 0.1905, 0.4753, 0.2322, 0.5538], ["connector_open", 0.4693, 0.426, 0.5025, 0.4927]], "s3-05_0025": [["connector_closed", 0.1918, 0.4696, 0.2399, 0.5532], ["connector_open", 0.4704, 0.428, 0.501, 0.4943]], "s3-05_0026": [["connector_open", 0.4706, 0.4271, 0.4993, 0.4937]], "s3-05_0031": [["connector_open", 0.4731, 0.4006, 0.5069, 0.4688]], "s3-05_0035": [["connector_closed", 0.2054, 0.4825, 0.2491, 0.5585], ["connector_closed", 0.4852, 0.3898, 0.5252, 0.4469]], "s3-05_0036": [["connector_closed", 0.2086, 0.4866, 0.2528, 0.5731], ["connector_closed", 0.4877, 0.3878, 0.5311, 0.4636]], "s3-05_0037": [["connector_closed", 0.2134, 0.4956, 0.2589, 0.5771], ["connector_closed", 0.4909, 0.3943, 0.5341, 0.472]], "s3-06_0001": [["connector_closed", 0.4422, 0.3255, 0.4796, 0.403]], "s3-06_0002": [["connector_closed", 0.4393, 0.3136, 0.4809, 0.4017]], "s3-06_0003": [["connector_closed", 0.4425, 0.3084, 0.4805, 0.3899]], "s3-06_0020": [["connector_closed", 0.7859, 0.3665, 0.8344, 0.4594], ["connector_open", 0.4924, 0.4054, 0.5268, 0.4695]], "s3-06_0033": [["connector_closed", 0.7964, 0.3825, 0.8524, 0.4775], ["connector_open", 0.4815, 0.4162, 0.5366, 0.5237]], "s3-06_0034": [["connector_closed", 0.5002, 0.4496, 0.5353, 0.5182], ["connector_closed", 0.7971, 0.3842, 0.8453, 0.4823]], "s3-06_0035": [["connector_closed", 0.4992, 0.4598, 0.5363, 0.5344], ["connector_closed", 0.796, 0.3915, 0.8477, 0.4894]], "s3-07_0001": [["connector_closed", 0.4012, 0.4825, 0.4368, 0.5565]], "s3-07_0002": [["connector_closed", 0.3992, 0.4799, 0.4362, 0.5522]], "s3-07_0003": [["connector_closed", 0.3972, 0.4733, 0.4342, 0.55]], "s3-07_0004": [["connector_closed", 0.3951, 0.4717, 0.4345, 0.5449]], "s3-07_0005": [["connector_closed", 0.3955, 0.4717, 0.4352, 0.5437]], "s3-07_0028": [["connector_closed", 0.7725, 0.3817, 0.8236, 0.4683], ["connector_open", 0.4841, 0.4338, 0.5238, 0.5039]], "s3-07_0029": [["connector_closed", 0.7766, 0.3834, 0.8266, 0.4698], ["connector_open", 0.4911, 0.4288, 0.5293, 0.4982]], "s3-07_0030": [["connector_closed", 0.7746, 0.4222, 0.8238, 0.5083], ["connector_open", 0.4909, 0.4235, 0.5309, 0.4971]], "s3-07_0035": [["connector_closed", 0.7796, 0.4922, 0.829, 0.5771], ["connector_closed", 0.4855, 0.4362, 0.5226, 0.5079]], "s3-07_0036": [["connector_closed", 0.7771, 0.4961, 0.8305, 0.5856], ["connector_closed", 0.4832, 0.4392, 0.5222, 0.5185]], "s3-07_0037": [["connector_closed", 0.7752, 0.4961, 0.8266, 0.5858], ["connector_closed", 0.4805, 0.4394, 0.5187, 0.52]], "s3-08_0001": [["connector_closed", 0.626, 0.7359, 0.7164, 0.7827]], "s3-08_0002": [["connector_closed", 0.5842, 0.7482, 0.6632, 0.7851]], "s3-08_0003": [["connector_closed", 0.6054, 0.7542, 0.695, 0.798]], "s3-08_0004": [["connector_closed", 0.6056, 0.7579, 0.6984, 0.8061], ["connector_closed", 0.8625, 0.103, 0.9454, 0.157]], "s3-08_0015": [["connector_closed", 0.4425, 0.2202, 0.5582, 0.2846], ["connector_open", 0.5771, 0.5965, 0.6591, 0.6426]], "s3-08_0020": [["connector_closed", 0.4404, 0.2118, 0.5559, 0.2761], ["connector_open", 0.574, 0.585, 0.6775, 0.6371]], "s3-08_0021": [["connector_closed", 0.4435, 0.2083, 0.56, 0.2736], ["connector_open", 0.5807, 0.5899, 0.6805, 0.6318]], "s3-08_0037": [["connector_closed", 0.5643, 0.6181, 0.659, 0.6692], ["connector_closed", 0.6006, 0.2289, 0.7205, 0.2958]], "s3-08_0038": [["connector_closed", 0.563, 0.6212, 0.6572, 0.6708], ["connector_closed", 0.5987, 0.2318, 0.718, 0.2976]], "s3-08_0039": [["connector_closed", 0.5617, 0.6159, 0.6558, 0.6665], ["connector_closed", 0.5992, 0.2261, 0.7194, 0.2928]], "s3-09_0001": [["connector_closed", 0.2518, 0.3027, 0.3376, 0.348]], "s3-09_0002": [["connector_closed", 0.2593, 0.3032, 0.3428, 0.3491]], "s3-09_0003": [["connector_closed", 0.2602, 0.301, 0.3455, 0.3465]], "s3-09_0004": [["connector_closed", 0.2609, 0.2987, 0.3473, 0.3459]], "s3-09_0005": [["connector_closed", 0.2596, 0.2983, 0.3436, 0.3446]], "s3-09_0012": [["connector_closed", 0.3432, 0.6484, 0.4565, 0.7106], ["connector_open", 0.2458, 0.2888, 0.3319, 0.3355]], "s3-09_0013": [["connector_closed", 0.3484, 0.6428, 0.4599, 0.7047], ["connector_open", 0.2356, 0.2846, 0.3241, 0.33]], "s3-09_0014": [["connector_closed", 0.3531, 0.6397, 0.468, 0.7025], ["connector_open", 0.2372, 0.2842, 0.3261, 0.3289]], "s3-09_0015": [["connector_closed", 0.3781, 0.6267, 0.4886, 0.6883], ["connector_open", 0.2369, 0.2956, 0.3269, 0.3411]], "s3-09_0016": [["connector_closed", 0.4314, 0.6147, 0.5379, 0.6722], ["connector_open", 0.2548, 0.3104, 0.3392, 0.3584]], "s3-09_0017": [["connector_closed", 0.3975, 0.6247, 0.5045, 0.6841], ["connector_open", 0.2558, 0.3146, 0.3399, 0.3661]], "s3-09_0027": [["connector_closed", 0.3959, 0.6547, 0.5061, 0.7168], ["connector_open", 0.3244, 0.3242, 0.4037, 0.3678]], "s3-09_0028": [["connector_closed", 0.3801, 0.6535, 0.4926, 0.7146], ["connector_open", 0.3202, 0.326, 0.3967, 0.3673]], "s3-09_0029": [["connector_closed", 0.3638, 0.6594, 0.477, 0.7228], ["connector_open", 0.322, 0.3343, 0.395, 0.3778]], "s3-09_0030": [["connector_closed", 0.3454, 0.6717, 0.4579, 0.7348], ["connector_open", 0.3241, 0.3388, 0.4005, 0.3767]], "s3-09_0033": [["connector_closed", 0.3877, 0.7247, 0.5006, 0.7902]], "s3-09_0034": [["connector_closed", 0.3824, 0.3724, 0.4738, 0.4207], ["connector_closed", 0.4033, 0.7266, 0.518, 0.7914]], "s3-09_0035": [["connector_closed", 0.3743, 0.3594, 0.4669, 0.4091], ["connector_closed", 0.3966, 0.7108, 0.5105, 0.7763]], "s3-10_0001": [["connector_closed", 0.4886, 0.1353, 0.5916, 0.195]], "s3-10_0002": [["connector_closed", 0.4928, 0.13, 0.5967, 0.1891]], "s3-10_0003": [["connector_closed", 0.4886, 0.1238, 0.5978, 0.1898]], "s3-10_0004": [["connector_closed", 0.4922, 0.1201, 0.5966, 0.182]], "s3-10_0005": [["connector_closed", 0.4955, 0.1218, 0.5995, 0.182]], "s3-10_0006": [["connector_closed", 0.4867, 0.1251, 0.593, 0.1835]], "s3-10_0012": [["connector_closed", -0.0019, 0.7819, 0.1059, 0.8572], ["connector_open", 0.5168, 0.1365, 0.6116, 0.2025]], "s3-10_0013": [["connector_closed", -0.0021, 0.7802, 0.1062, 0.8552], ["connector_open", 0.4896, 0.1255, 0.5926, 0.194]], "s3-10_0014": [["connector_closed", -0.0024, 0.7757, 0.1066, 0.8509], ["connector_open", 0.4827, 0.1292, 0.5885, 0.1842]], "s3-10_0036": [["connector_closed", 0.33, 0.6405, 0.48, 0.7222], ["connector_closed", 0.386, 0.2066, 0.4878, 0.2573]], "s3-10_0037": [["connector_closed", 0.3831, 0.2071, 0.4886, 0.259], ["connector_closed", 0.3411, 0.6566, 0.4327, 0.7016]], "s3-10_0038": [["connector_closed", 0.3651, 0.2054, 0.4669, 0.2553], ["connector_closed", 0.2164, 0.645, 0.3262, 0.7024]], "s3-10_0039": [["connector_closed", 0.3647, 0.2011, 0.4702, 0.2557], ["connector_closed", 0.2214, 0.6423, 0.3581, 0.7148]], "s3-10_0040": [["connector_closed", 0.223, 0.6393, 0.3632, 0.7169], ["connector_closed", 0.3689, 0.2021, 0.4745, 0.2558]], "s3-10_0041": [["connector_closed", 0.2274, 0.6406, 0.3691, 0.7191], ["connector_closed", 0.3767, 0.2074, 0.484, 0.2626]], "s3-11_0001": [["connector_closed", 0.4442, 0.1689, 0.5507, 0.2291]], "s3-11_0002": [["connector_closed", 0.4481, 0.1659, 0.5533, 0.2237]], "s3-11_0003": [["connector_closed", 0.4598, 0.1623, 0.5614, 0.2208]], "s3-11_0004": [["connector_closed", 0.4673, 0.1619, 0.565, 0.2182]], "s3-11_0005": [["connector_closed", 0.4676, 0.1655, 0.5654, 0.2183]], "s3-11_0015": [["connector_closed", 0.3223, 0.6098, 0.4155, 0.6702], ["connector_open", 0.4146, 0.2097, 0.496, 0.2592]], "s3-11_0018": [["connector_closed", 0.3328, 0.6019, 0.4594, 0.6719], ["connector_closed", 0.4484, 0.2049, 0.545, 0.2525]], "s3-11_0019": [["connector_closed", 0.3424, 0.597, 0.4679, 0.6651], ["connector_closed", 0.4647, 0.2077, 0.5611, 0.2566]], "s3-11_0020": [["connector_closed", 0.3448, 0.596, 0.4728, 0.6669], ["connector_closed", 0.4668, 0.209, 0.5607, 0.2555]], "s3-11_0021": [["connector_closed", 0.3501, 0.5993, 0.477, 0.6684], ["connector_closed", 0.4635, 0.2113, 0.5345, 0.255]], "s3-11_0027": [["connector_closed", 0.3608, 0.6202, 0.4888, 0.6913], ["connector_open", 0.4877, 0.224, 0.5723, 0.2768]], "s3-11_0028": [["connector_closed", 0.3618, 0.6219, 0.4903, 0.693], ["connector_open", 0.4869, 0.2269, 0.5798, 0.2917]], "s3-11_0029": [["connector_closed", 0.3611, 0.6171, 0.4876, 0.688], ["connector_open", 0.4895, 0.2257, 0.5755, 0.2753]], "s3-11_0034": [["connector_closed", 0.4797, 0.2233, 0.5795, 0.2751], ["connector_closed", 0.3539, 0.6156, 0.4777, 0.6801]], "s3-11_0035": [["connector_closed", 0.2958, 0.6119, 0.4178, 0.6806], ["connector_closed", 0.4758, 0.2246, 0.5793, 0.2779]], "s3-11_0036": [["connector_closed", 0.2934, 0.6089, 0.4191, 0.6777], ["connector_closed", 0.4774, 0.223, 0.5796, 0.2765]], "s3-12_0001": [["connector_closed", 0.3882, 0.7674, 0.4953, 0.8337]], "s3-12_0002": [["connector_closed", 0.3864, 0.7696, 0.4958, 0.8359]], "s3-12_0003": [["connector_closed", 0.3776, 0.7708, 0.4864, 0.8382]], "s3-12_0004": [["connector_closed", 0.3783, 0.772, 0.4878, 0.8362]], "s3-12_0013": [["connector_closed", 0.5167, 0.1122, 0.663, 0.1894], ["connector_open", 0.2823, 0.6599, 0.3932, 0.7233]], "s3-12_0014": [["connector_closed", 0.4711, 0.1429, 0.6124, 0.2217], ["connector_open", 0.3014, 0.6404, 0.4098, 0.7034]], "s3-12_0015": [["connector_closed", 0.4042, 0.1391, 0.5523, 0.2236], ["connector_open", 0.3056, 0.6414, 0.4113, 0.7029]], "s3-12_0016": [["connector_closed", 0.4008, 0.1409, 0.5423, 0.2244], ["connector_open", 0.3129, 0.6438, 0.4161, 0.7005]], "s3-12_0026": [["connector_closed", 0.4293, 0.1223, 0.5763, 0.2049], ["connector_open", 0.3428, 0.6316, 0.4293, 0.6751]], "s3-12_0027": [["connector_open", 0.3281, 0.6255, 0.4398, 0.6933], ["connector_closed", 0.4259, 0.1224, 0.5723, 0.2056]], "s3-12_0028": [["connector_open", 0.3265, 0.6246, 0.4324, 0.6846], ["connector_closed", 0.4243, 0.1194, 0.5698, 0.2017]], "s3-12_0029": [["connector_closed", 0.4206, 0.1184, 0.5675, 0.1997], ["connector_open", 0.3219, 0.6303, 0.4235, 0.6854]], "s3-12_0030": [["connector_closed", 0.418, 0.1193, 0.562, 0.1998], ["connector_open", 0.3183, 0.632, 0.416, 0.6864]], "s3-12_0031": [["connector_closed", 0.4062, 0.1129, 0.5527, 0.1974], ["connector_open", 0.32, 0.6248, 0.424, 0.6806]], "s3-12_0032": [["connector_closed", 0.4117, 0.1117, 0.5566, 0.1952], ["connector_open", 0.3251, 0.6236, 0.4208, 0.6798]], "s3-12_0033": [["connector_closed", 0.4152, 0.1112, 0.5604, 0.1951], ["connector_open", 0.326, 0.6224, 0.4271, 0.6772]], "s3-12_0036": [["connector_closed", 0.321, 0.6197, 0.4288, 0.6851], ["connector_closed", 0.4094, 0.1111, 0.5562, 0.1962]], "s3-12_0037": [["connector_closed", 0.4095, 0.1106, 0.5559, 0.1957], ["connector_closed", 0.322, 0.6162, 0.4309, 0.6818]], "s3-13_0001": [["connector_closed", 0.5281, 0.774, 0.6591, 0.8521]], "s3-13_0002": [["connector_closed", 0.5263, 0.7751, 0.6588, 0.8523]], "s3-13_0003": [["connector_closed", 0.5238, 0.7741, 0.6536, 0.8515]], "s3-13_0004": [["connector_closed", 0.5214, 0.7734, 0.6518, 0.8499]], "s3-13_0005": [["connector_closed", 0.5111, 0.7709, 0.6431, 0.8475]], "s3-13_0022": [["connector_closed", 0.5136, 0.0844, 0.6831, 0.185], ["connector_open", 0.5018, 0.7042, 0.5456, 0.737]], "s3-13_0023": [["connector_closed", 0.516, 0.0832, 0.6905, 0.1844], ["connector_open", 0.4297, 0.683, 0.5498, 0.7573]], "s3-13_0028": [["connector_closed", 0.5314, 0.0776, 0.7225, 0.1786], ["connector_open", 0.4205, 0.6816, 0.5264, 0.7439]], "s3-13_0035": [["connector_closed", 0.5042, 0.0795, 0.6763, 0.1798], ["connector_closed", 0.4098, 0.6778, 0.5355, 0.7476]], "s3-13_0036": [["connector_closed", 0.5125, 0.0797, 0.6792, 0.1786], ["connector_closed", 0.4156, 0.6706, 0.5434, 0.743]], "s3-13_0037": [["connector_closed", 0.5014, 0.0758, 0.6759, 0.1755], ["connector_closed", 0.4036, 0.6712, 0.5296, 0.7424]], "s3-14_0001": [["connector_closed", 0.6759, 0.6111, 0.733, 0.7159]], "s3-14_0002": [["connector_closed", 0.6782, 0.6142, 0.7361, 0.7207]], "s3-14_0003": [["connector_closed", 0.6787, 0.6128, 0.7339, 0.7162]], "s3-14_0004": [["connector_closed", 0.6792, 0.6126, 0.7345, 0.7156], ["connector_closed", 0.4357, -0.0046, 0.4982, 0.1009]], "s3-14_0005": [["connector_closed", 0.6811, 0.6123, 0.7378, 0.7135], ["connector_closed", 0.406, -0.0043, 0.4684, 0.0889]], "s3-14_0006": [["connector_closed", 0.6847, 0.6084, 0.7409, 0.7105]], "s3-14_0007": [["connector_closed", 0.6841, 0.609, 0.7418, 0.7145]], "s3-14_0031": [["connector_closed", 0.1952, 0.4288, 0.2572, 0.5446], ["connector_open", 0.6044, 0.5232, 0.6547, 0.6167]], "s3-14_0032": [["connector_open", 0.6037, 0.5255, 0.6549, 0.6174], ["connector_closed", 0.1938, 0.4317, 0.2549, 0.5462]], "s3-14_0033": [["connector_closed", 0.1948, 0.4326, 0.2537, 0.5433], ["connector_open", 0.6054, 0.53, 0.656, 0.6197]], "s3-14_0034": [["connector_closed", 0.1954, 0.4315, 0.2541, 0.5417], ["connector_open", 0.6028, 0.5281, 0.651, 0.6161]], "s3-14_0035": [["connector_open", 0.6092, 0.5286, 0.659, 0.6192], ["connector_closed", 0.2055, 0.4355, 0.2688, 0.5495]], "s3-14_0036": [["connector_closed", 0.2012, 0.4401, 0.2652, 0.558], ["connector_open", 0.6139, 0.544, 0.6601, 0.6265]], "s3-14_0040": [["connector_closed", 0.6113, 0.5365, 0.6677, 0.6351], ["connector_closed", 0.2057, 0.4389, 0.2701, 0.5567]], "s3-14_0041": [["connector_closed", 0.6139, 0.5369, 0.6685, 0.6328], ["connector_closed", 0.206, 0.4392, 0.2715, 0.5566]], "s3-14_0042": [["connector_closed", 0.6161, 0.5337, 0.6697, 0.6308], ["connector_closed", 0.2049, 0.4321, 0.2721, 0.5568]], "s3-15_0001": [["connector_closed", 0.7025, 0.5646, 0.7582, 0.6701]], "s3-15_0002": [["connector_closed", 0.7031, 0.5705, 0.7571, 0.6716]], "s3-15_0003": [["connector_closed", 0.6981, 0.5654, 0.7542, 0.6703]], "s3-15_0004": [["connector_closed", 0.7045, 0.5698, 0.7596, 0.6707]], "s3-15_0005": [["connector_closed", 0.7102, 0.5653, 0.763, 0.6666]], "s3-15_0006": [["connector_closed", 0.7058, 0.5568, 0.7588, 0.6554], ["connector_closed", 0.1744, 0.3845, 0.2081, 0.4506]], "s3-15_0007": [["connector_closed", 0.7042, 0.552, 0.757, 0.648]], "s3-15_0026": [["connector_open", 0.6894, 0.4938, 0.7388, 0.5867], ["connector_closed", 0.2894, 0.4962, 0.3506, 0.6047]], "s3-15_0027": [["connector_closed", 0.2927, 0.4977, 0.349, 0.5993], ["connector_open", 0.6837, 0.5206, 0.7325, 0.6089]], "s3-15_0028": [["connector_closed", 0.2943, 0.5011, 0.3501, 0.6075], ["connector_open", 0.6874, 0.5271, 0.7336, 0.6122]], "s3-15_0029": [["connector_closed", 0.2982, 0.5103, 0.3548, 0.6182], ["connector_open", 0.683, 0.5305, 0.7337, 0.6109]], "s3-15_0031": [["connector_open", 0.6834, 0.5325, 0.7335, 0.6219], ["connector_closed", 0.2935, 0.5109, 0.3471, 0.6061]], "s3-15_0034": [["connector_closed", 0.6724, 0.569, 0.7307, 0.6713], ["connector_closed", 0.2944, 0.3674, 0.35, 0.4748]], "s3-15_0035": [["connector_closed", 0.6765, 0.5715, 0.7325, 0.6763], ["connector_closed", 0.2948, 0.3687, 0.3498, 0.4763]], "s2-01_0001": [["connector_closed", 0.81, 0.4233, 0.8549, 0.5039]], "s2-01_0002": [["connector_closed", 0.8078, 0.4079, 0.8524, 0.4856]], "s2-01_0003": [["connector_closed", 0.8009, 0.3924, 0.847, 0.4722]], "s2-01_0004": [["connector_closed", 0.7987, 0.3826, 0.8416, 0.4606]], "s2-01_0005": [["connector_closed", 0.7917, 0.3771, 0.8365, 0.4547]], "s2-01_0006": [["connector_closed", 0.7766, 0.3925, 0.8212, 0.4678]], "s2-01_0007": [["connector_closed", 0.7659, 0.4101, 0.8109, 0.4875]], "s2-01_0008": [["connector_closed", 0.7595, 0.4116, 0.8019, 0.4861]], "s2-01_0012": [["connector_closed", 0.7632, 0.4182, 0.8065, 0.4961]], "s2-01_0025": [["connector_open", 0.502, 0.4324, 0.5382, 0.5086]], "s2-01_0026": [["connector_open", 0.4965, 0.4242, 0.532, 0.507]], "s2-01_0027": [["connector_open", 0.4883, 0.4529, 0.5272, 0.5302]], "s2-01_0028": [["connector_open", 0.4857, 0.4993, 0.5222, 0.5804]], "s2-01_0029": [["connector_open", 0.4785, 0.5332, 0.5191, 0.6187]], "s2-01_0030": [["connector_open", 0.4855, 0.5429, 0.5256, 0.6214]], "s2-01_0031": [["connector_open", 0.4931, 0.5414, 0.5325, 0.6208]], "s2-01_0032": [["connector_open", 0.4948, 0.5283, 0.5324, 0.6112]], "s2-01_0033": [["connector_open", 0.4956, 0.5269, 0.5348, 0.6056]], "s2-01_0034": [["connector_open", 0.5014, 0.5218, 0.5369, 0.5958]], "s2-01_0035": [["connector_open", 0.506, 0.519, 0.5376, 0.579]], "s2-01_0036": [["connector_open", 0.5027, 0.5237, 0.5389, 0.594]], "s2-01_0037": [["connector_open", 0.4995, 0.5275, 0.5397, 0.6053]], "s2-01_0038": [["connector_open", 0.4983, 0.5329, 0.5393, 0.6112]], "s2-01_0039": [["connector_open", 0.4975, 0.5521, 0.5379, 0.6318]], "s2-01_0040": [["connector_open", 0.4986, 0.5538, 0.5376, 0.6318]], "s2-01_0041": [["connector_open", 0.5006, 0.5522, 0.5353, 0.6261]], "s2-01_0042": [["connector_open", 0.4983, 0.5569, 0.5335, 0.6293]], "s2-01_0043": [["connector_open", 0.499, 0.5589, 0.5338, 0.6308]], "s2-01_0044": [["connector_open", 0.5002, 0.5639, 0.5338, 0.6324]], "s2-01_0045": [["connector_open", 0.4987, 0.5638, 0.5333, 0.634]], "s2-01_0046": [["connector_open", 0.4998, 0.5614, 0.5348, 0.6333]], "s2-01_0047": [["connector_open", 0.4881, 0.5519, 0.5293, 0.6287]], "s2-01_0048": [["connector_open", 0.4862, 0.5536, 0.5263, 0.6402]], "s2-01_0049": [["connector_open", 0.4936, 0.5563, 0.5291, 0.6319]], "s2-01_0052": [["connector_open", 0.4862, 0.5415, 0.5291, 0.6197]], "s2-01_0053": [["connector_open", 0.4812, 0.5612, 0.5204, 0.6328]], "s2-01_0054": [["connector_open", 0.4844, 0.5653, 0.5281, 0.6471]], "s2-01_0055": [["connector_open", 0.4853, 0.5733, 0.5207, 0.6463]], "s2-01_0056": [["connector_open", 0.4844, 0.5765, 0.5197, 0.6495]], "s2-01_0061": [["connector_closed", 0.4765, 0.5759, 0.515, 0.655]], "s2-01_0062": [["connector_closed", 0.4712, 0.5711, 0.5124, 0.654]], "s2-01_0063": [["connector_closed", 0.4698, 0.5658, 0.5101, 0.6471]], "s2-02_0001": [["connector_closed", 0.4654, 0.6547, 0.5412, 0.6961]], "s2-02_0002": [["connector_closed", 0.4882, 0.6577, 0.5622, 0.7019]], "s2-02_0003": [["connector_closed", 0.502, 0.6544, 0.58, 0.6981]], "s2-02_0004": [["connector_closed", 0.5098, 0.658, 0.5846, 0.7004]], "s2-02_0005": [["connector_closed", 0.5188, 0.6523, 0.6008, 0.6968]], "s2-02_0006": [["connector_closed", 0.5369, 0.645, 0.6116, 0.6873]], "s2-02_0008": [["connector_closed", 0.5643, 0.6468, 0.638, 0.6874]], "s2-02_0009": [["connector_closed", 0.5612, 0.6501, 0.6353, 0.6931]], "s2-02_0010": [["connector_closed", 0.5316, 0.6559, 0.6072, 0.6975]], "s2-02_0011": [["connector_closed", 0.5263, 0.6618, 0.6047, 0.704]], "s2-02_0012": [["connector_closed", 0.5316, 0.6631, 0.6093, 0.7065]], "s2-02_0013": [["connector_closed", 0.5494, 0.6492, 0.6148, 0.6873]], "s2-02_0033": [["connector_open", 0.3307, 0.4978, 0.418, 0.5406]], "s2-02_0034": [["connector_open", 0.337, 0.4989, 0.4195, 0.5406]], "s2-02_0036": [["connector_open", 0.421, 0.594, 0.5156, 0.6463]], "s2-02_0037": [["connector_open", 0.4493, 0.5979, 0.5494, 0.6514]], "s2-02_0038": [["connector_open", 0.5059, 0.6248, 0.5736, 0.6585]], "s2-02_0039": [["connector_open", 0.5295, 0.6298, 0.6012, 0.6645]], "s2-02_0040": [["connector_open", 0.4594, 0.6099, 0.5516, 0.6572]], "s2-02_0041": [["connector_open", 0.5211, 0.5719, 0.6052, 0.6175]], "s2-02_0042": [["connector_open", 0.5218, 0.5967, 0.6086, 0.6368]], "s2-02_0043": [["connector_open", 0.5005, 0.5896, 0.6122, 0.6492]], "s2-02_0044": [["connector_open", 0.3749, 0.5262, 0.4487, 0.5729]], "s2-02_0046": [["connector_open", 0.3658, 0.4465, 0.4513, 0.4882]], "s2-02_0047": [["connector_open", 0.4004, 0.4562, 0.4523, 0.4895]], "s2-02_0048": [["connector_open", 0.3916, 0.495, 0.4632, 0.5316]], "s2-02_0056": [["connector_closed", 0.431, 0.4579, 0.5039, 0.4907]], "s2-02_0057": [["connector_closed", 0.4132, 0.45, 0.4888, 0.488]], "s2-02_0058": [["connector_closed", 0.3807, 0.4334, 0.4544, 0.4727]], "s2-02_0059": [["connector_closed", 0.3916, 0.4259, 0.4679, 0.4653]], "s2-02_0060": [["connector_closed", 0.4198, 0.4267, 0.4924, 0.467]], "s2-03_0001": [["connector_closed", 0.3302, 0.6453, 0.4555, 0.7178]], "s2-03_0002": [["connector_closed", 0.3309, 0.6457, 0.4559, 0.7174]], "s2-03_0009": [["connector_open", 0.0045, 0.6183, 0.1516, 0.6995]], "s2-03_0010": [["connector_open", 0.3859, 0.6131, 0.5426, 0.7035]], "s2-03_0011": [["connector_open", 0.3673, 0.6107, 0.5504, 0.7141]], "s2-03_0012": [["connector_open", 0.4229, 0.6667, 0.6025, 0.766]], "s2-03_0013": [["connector_open", 0.4712, 0.699, 0.6594, 0.7981]], "s2-03_0014": [["connector_open", 0.4782, 0.7159, 0.6699, 0.8095]], "s2-03_0015": [["connector_open", 0.396, 0.6583, 0.5861, 0.7612]], "s2-03_0016": [["connector_open", 0.4057, 0.6672, 0.5929, 0.7756]], "s2-03_0017": [["connector_open", 0.5829, 0.6735, 0.6498, 0.7104]], "s2-03_0018": [["connector_open", 0.1952, 0.671, 0.3962, 0.7741]], "s2-03_0020": [["connector_open", 0.0005, 0.7458, 0.0995, 0.8539]], "s2-03_0021": [["connector_open", 0.0006, 0.7429, 0.2015, 0.8685]], "s2-03_0022": [["connector_open", 0.0027, 0.7498, 0.2063, 0.8758]], "s2-03_0023": [["connector_open", 0.0571, 0.7983, 0.284, 0.9249]], "s2-03_0024": [["connector_open", 0.1148, 0.8674, 0.3265, 0.9858]], "s2-03_0025": [["connector_open", 0.178, 0.9139, 0.3693, 1.0]], "s2-03_0027": [["connector_open", 0.0039, 0.8827, 0.2125, 0.9995]], "s2-03_0040": [["connector_closed", 0.1905, 0.521, 0.3692, 0.6286]], "s2-03_0041": [["connector_closed", 0.1625, 0.6399, 0.2935, 0.7191]], "s2-03_0042": [["connector_closed", 0.1947, 0.6525, 0.3065, 0.7257]], "s2-03_0043": [["connector_closed", 0.1633, 0.6375, 0.263, 0.6877]], "s2-03_0054": [["connector_closed", 0.0, 0.6226, 0.1286, 0.7001]], "s2-03_0055": [["connector_closed", 0.0, 0.6234, 0.1288, 0.7002]], "s2-03_0056": [["connector_closed", 0.0, 0.6232, 0.1275, 0.6998]], "s2-03_0057": [["connector_closed", 0.0011, 0.6231, 0.1282, 0.6994]], "s2-03_0058": [["connector_closed", 0.0, 0.6218, 0.1264, 0.7001]], "s2-04_0001": [["connector_closed", 0.2885, 0.5677, 0.4996, 0.6904]], "s2-04_0014": [["connector_open", 0.2873, 0.5386, 0.5013, 0.657]], "s2-04_0016": [["connector_open", 0.3422, 0.5001, 0.5373, 0.6181]], "s2-04_0018": [["connector_open", 0.3041, 0.4867, 0.5038, 0.6086]], "s2-04_0019": [["connector_open", 0.3599, 0.5143, 0.4779, 0.5636]], "s2-04_0020": [["connector_open", 0.256, 0.4668, 0.4881, 0.6107]], "s2-04_0021": [["connector_open", 0.3727, 0.5324, 0.4666, 0.6283]], "s2-04_0022": [["connector_open", 0.3261, 0.5042, 0.4753, 0.5809]], "s2-04_0023": [["connector_open", 0.311, 0.5107, 0.5195, 0.6362]], "s2-04_0027": [["connector_open", 0.3144, 0.5143, 0.4875, 0.5765]], "s2-04_0028": [["connector_open", 0.3342, 0.5057, 0.558, 0.627]], "s2-04_0029": [["connector_open", 0.3389, 0.5295, 0.5127, 0.6023]], "s2-04_0031": [["connector_closed", 0.2896, 0.5112, 0.5385, 0.6409]], "s2-04_0032": [["connector_closed", 0.2826, 0.5222, 0.518, 0.6418]], "s2-04_0033": [["connector_closed", 0.2945, 0.5273, 0.5221, 0.6446]], "s2-04_0034": [["connector_closed", 0.2755, 0.5193, 0.5014, 0.634]], "s2-04_0035": [["connector_closed", 0.2887, 0.5199, 0.5023, 0.6283]], "s2-04_0036": [["connector_closed", 0.2715, 0.4958, 0.4989, 0.6115]], "s2-04_0037": [["connector_closed", 0.2704, 0.513, 0.4916, 0.6303]], "s2-05_0013": [["connector_open", 0.3053, 0.4608, 0.5386, 0.5726]], "s2-05_0015": [["connector_open", 0.2324, 0.4363, 0.4778, 0.5759]], "s2-05_0016": [["connector_open", 0.3018, 0.4543, 0.5175, 0.5857]], "s2-05_0017": [["connector_open", 0.3541, 0.4852, 0.513, 0.572]], "s2-05_0018": [["connector_open", 0.4756, 0.5228, 0.6693, 0.6581]], "s2-05_0019": [["connector_open", 0.4741, 0.5547, 0.6226, 0.6333]], "s2-05_0022": [["connector_closed", 0.3656, 0.4881, 0.605, 0.62]], "s2-05_0023": [["connector_closed", 0.3204, 0.4648, 0.5576, 0.6012]], "s2-05_0024": [["connector_closed", 0.3673, 0.48, 0.6035, 0.6184]], "s2-05_0025": [["connector_closed", 0.3763, 0.4845, 0.6067, 0.6178]], "s2-06_0001": [["connector_closed", 0.2763, 0.5251, 0.4645, 0.6363]], "s2-06_0002": [["connector_closed", 0.2292, 0.5645, 0.423, 0.6775]], "s2-06_0003": [["connector_closed", 0.0005, 0.7483, 0.1471, 0.8488]], "s2-06_0004": [["connector_closed", 0.1508, 0.6831, 0.3531, 0.7981]], "s2-06_0014": [["connector_closed", 0.1738, 0.4244, 0.4045, 0.5581]], "s2-06_0015": [["connector_closed", 0.0585, 0.5097, 0.2531, 0.6171]], "s2-06_0016": [["connector_closed", 0.2455, 0.5274, 0.4139, 0.6254]], "s2-06_0020": [["connector_closed", 0.2033, 0.4513, 0.3988, 0.5582]], "s2-06_0021": [["connector_closed", 0.2551, 0.4522, 0.454, 0.5714]], "s2-06_0022": [["connector_closed", 0.2519, 0.4471, 0.4495, 0.5652]], "s2-06_0023": [["connector_closed", 0.2449, 0.462, 0.4356, 0.5735]], "s2-07_0001": [["connector_closed", 0.1844, 0.4295, 0.4247, 0.5693]], "s2-07_0006": [["connector_open", 0.1006, 0.3967, 0.3341, 0.5252]], "s2-07_0007": [["connector_open", 0.1951, 0.4616, 0.423, 0.584]], "s2-07_0008": [["connector_open", 0.2544, 0.4623, 0.494, 0.5993]], "s2-07_0009": [["connector_open", 0.2382, 0.4566, 0.4655, 0.5838]], "s2-07_0010": [["connector_open", 0.2508, 0.4621, 0.4851, 0.5963]], "s2-07_0011": [["connector_open", 0.2982, 0.4863, 0.5318, 0.6184]], "s2-07_0012": [["connector_open", 0.3259, 0.4921, 0.5558, 0.6248]], "s2-07_0017": [["connector_closed", 0.146, 0.4559, 0.3762, 0.5825]], "s2-07_0018": [["connector_closed", 0.0332, 0.4554, 0.2678, 0.5799]], "s2-07_0019": [["connector_closed", 0.0011, 0.4647, 0.1981, 0.5928]], "s2-07_0020": [["connector_closed", 0.0636, 0.4662, 0.2965, 0.5984]], "s2-08_0001": [["connector_closed", 0.1955, 0.4555, 0.4127, 0.5782]], "s2-08_0002": [["connector_closed", 0.1072, 0.4599, 0.3233, 0.5815]], "s2-08_0003": [["connector_closed", 0.0588, 0.5022, 0.2637, 0.6218]], "s2-08_0007": [["connector_open", 0.2507, 0.3752, 0.4814, 0.505]], "s2-08_0008": [["connector_open", 0.1444, 0.3378, 0.3403, 0.4575]], "s2-08_0009": [["connector_open", 0.2152, 0.3834, 0.4269, 0.5022]], "s2-08_0010": [["connector_open", 0.319, 0.4136, 0.5411, 0.5354]], "s2-08_0011": [["connector_open", 0.2987, 0.4242, 0.5179, 0.5435]], "s2-08_0012": [["connector_open", 0.375, 0.4164, 0.6048, 0.5388]], "s2-08_0015": [["connector_closed", 0.0573, 0.3632, 0.3041, 0.5029]], "s2-08_0016": [["connector_closed", 0.1544, 0.4007, 0.3268, 0.5189]], "s2-08_0017": [["connector_closed", 0.1909, 0.4197, 0.3983, 0.5337]], "s2-08_0018": [["connector_closed", 0.1385, 0.4153, 0.3748, 0.5454]], "s2-08_0019": [["connector_closed", 0.1928, 0.3966, 0.4291, 0.5236]], "s2-08_0020": [["connector_closed", 0.1722, 0.4051, 0.4117, 0.5298]], "s2-09_0001": [["connector_closed", 0.1705, 0.3687, 0.413, 0.4948]], "s2-09_0002": [["connector_closed", 0.1897, 0.3826, 0.4262, 0.5097]], "s2-09_0003": [["connector_closed", 0.2135, 0.4309, 0.4441, 0.537]], "s2-09_0004": [["connector_closed", 0.3367, 0.4965, 0.5345, 0.6005]], "s2-09_0005": [["connector_closed", 0.1848, 0.4236, 0.4001, 0.5401]], "s2-09_0006": [["connector_closed", 0.0567, 0.3484, 0.2054, 0.4528]], "s2-09_0007": [["connector_closed", 0.1658, 0.4018, 0.3718, 0.5134]], "s2-09_0008": [["connector_closed", 0.0952, 0.4089, 0.3123, 0.5224]], "s2-09_0009": [["connector_closed", 0.1981, 0.4027, 0.4147, 0.5226]], "s2-09_0010": [["connector_closed", 0.2411, 0.4616, 0.4903, 0.5738]], "s2-09_0012": [["connector_closed", 0.237, 0.3963, 0.4136, 0.5196]], "s2-09_0013": [["connector_open", 0.2387, 0.4483, 0.4186, 0.5683]], "s2-09_0014": [["connector_open", 0.3439, 0.461, 0.488, 0.561]], "s2-09_0015": [["connector_open", 0.1425, 0.4194, 0.3041, 0.5262]], "s2-09_0016": [["connector_open", 0.2905, 0.4285, 0.4375, 0.5385]], "s2-09_0017": [["connector_open", 0.3128, 0.4701, 0.5249, 0.5702]], "s2-09_0018": [["connector_open", 0.3035, 0.4151, 0.5096, 0.5213]], "s2-09_0020": [["connector_open", 0.3877, 0.4599, 0.4962, 0.5052]], "s2-09_0021": [["connector_open", 0.3308, 0.3932, 0.5329, 0.5214]], "s2-09_0022": [["connector_open", 0.4594, 0.4744, 0.583, 0.5184]], "s2-09_0023": [["connector_open", 0.4773, 0.539, 0.5769, 0.5902]], "s2-09_0024": [["connector_open", 0.3578, 0.6257, 0.4778, 0.7035]], "s2-09_0025": [["connector_open", 0.3759, 0.6918, 0.4454, 0.7387]], "s2-09_0026": [["connector_open", 0.4801, 0.8201, 0.5377, 0.8609]], "s2-09_0027": [["connector_open", 0.1432, 0.6506, 0.196, 0.6975]], "s2-09_0028": [["connector_open", 0.196, 0.5393, 0.2567, 0.6262]], "s2-09_0029": [["connector_open", 0.0499, 0.5038, 0.1867, 0.5741]], "s2-09_0030": [["connector_open", 0.1831, 0.5976, 0.259, 0.6568]], "s2-09_0031": [["connector_open", 0.2653, 0.6912, 0.339, 0.7466]], "s2-09_0032": [["connector_open", 0.25, 0.6961, 0.3326, 0.7493]], "s2-09_0033": [["connector_open", 0.2914, 0.6557, 0.4206, 0.7283]], "s2-09_0034": [["connector_open", 0.4981, 0.6679, 0.5836, 0.7362]], "s2-09_0037": [["connector_closed", 0.1869, 0.5686, 0.3289, 0.6517]], "s2-09_0038": [["connector_closed", 0.2055, 0.6554, 0.3367, 0.7277]], "s2-10_0001": [["connector_closed", 0.4072, 0.67, 0.5394, 0.748]], "s2-10_0002": [["connector_closed", 0.4087, 0.6685, 0.5394, 0.7455]], "s2-10_0003": [["connector_closed", 0.5262, 0.6523, 0.7286, 0.7677]], "s2-10_0007": [["connector_open", 0.4026, 0.5399, 0.6266, 0.663]], "s2-10_0008": [["connector_open", 0.3605, 0.536, 0.6029, 0.6616]], "s2-10_0009": [["connector_open", 0.3601, 0.5489, 0.623, 0.6873]], "s2-10_0012": [["connector_open", 0.2882, 0.5374, 0.5857, 0.7031]], "s2-10_0013": [["connector_open", 0.1708, 0.4566, 0.4442, 0.609]], "s2-10_0016": [["connector_open", 0.5528, 0.5495, 0.6999, 0.6034]], "s2-10_0017": [["connector_open", 0.5019, 0.5452, 0.6563, 0.6047]], "s2-10_0018": [["connector_open", 0.5461, 0.5678, 0.7204, 0.6295]], "s2-10_0019": [["connector_open", 0.5587, 0.5536, 0.7864, 0.6954]], "s2-10_0020": [["connector_open", 0.6343, 0.5711, 0.8048, 0.6288]], "s2-10_0021": [["connector_open", 0.6006, 0.583, 0.782, 0.6823]], "s2-10_0022": [["connector_open", 0.505, 0.6174, 0.7387, 0.7428]], "s2-10_0023": [["connector_open", 0.2329, 0.5012, 0.4898, 0.6585]], "s2-10_0024": [["connector_open", 0.0532, 0.5536, 0.2692, 0.6928]], "s2-10_0026": [["connector_open", 0.3813, 0.4335, 0.5761, 0.5682]], "s2-10_0032": [["connector_closed", 0.0, 0.3574, 0.2027, 0.5062]], "s2-10_0033": [["connector_closed", 0.1761, 0.4201, 0.42, 0.5529]], "s2-10_0034": [["connector_closed", 0.0062, 0.4396, 0.2549, 0.5803]], "s2-10_0035": [["connector_closed", 0.1087, 0.4074, 0.3513, 0.5381]], "s2-10_0036": [["connector_closed", 0.0407, 0.3468, 0.3008, 0.4871]], "s2-10_0037": [["connector_closed", 0.2217, 0.3856, 0.4888, 0.5316]], "s2-10_0038": [["connector_closed", 0.3267, 0.384, 0.6103, 0.5366]], "s2-10_0039": [["connector_closed", 0.2035, 0.3913, 0.4969, 0.5476]], "s2-10_0040": [["connector_closed", 0.2825, 0.4471, 0.5758, 0.6072]], "s2-10_0041": [["connector_closed", 0.3046, 0.4734, 0.5647, 0.6323]], "s2-10_0042": [["connector_closed", 0.2615, 0.4728, 0.5163, 0.6374]], "s2-10_0043": [["connector_closed", 0.1041, 0.3973, 0.3871, 0.553]], "s2-10_0044": [["connector_closed", 0.0, 0.2544, 0.2008, 0.3981]], "s2-10_0045": [["connector_closed", 0.1306, 0.3318, 0.3694, 0.4679]], "s2-10_0046": [["connector_closed", 0.2548, 0.4302, 0.4919, 0.5705]], "s2-10_0047": [["connector_closed", 0.186, 0.4237, 0.4212, 0.567]], "s2-10_0048": [["connector_closed", 0.174, 0.4131, 0.4016, 0.5519]], "s2-10_0049": [["connector_closed", 0.2196, 0.4272, 0.4427, 0.5641]], "s2-10_0050": [["connector_closed", 0.212, 0.4331, 0.4477, 0.5679]], "s2-11_0001": [["connector_closed", 0.4234, 0.5148, 0.574, 0.6008]], "s2-11_0002": [["connector_closed", 0.4244, 0.5131, 0.5738, 0.5997]], "s2-11_0003": [["connector_closed", 0.5822, 0.4993, 0.7201, 0.576]], "s2-11_0004": [["connector_closed", 0.2336, 0.7115, 0.4668, 0.8252]], "s2-11_0005": [["connector_closed", 0.2326, 0.3423, 0.4631, 0.4583]], "s2-11_0006": [["connector_closed", 0.2987, 0.4429, 0.5378, 0.5595]], "s2-11_0007": [["connector_closed", 0.4037, 0.4522, 0.6141, 0.5759]], "s2-11_0008": [["connector_closed", 0.0, 0.4699, 0.1426, 0.5794]], "s2-11_0009": [["connector_open", 0.0, 0.4509, 0.1497, 0.5694]], "s2-11_0019": [["connector_open", 0.0, 0.4151, 0.1823, 0.5472]], "s2-11_0021": [["connector_open", 0.0127, 0.3479, 0.3207, 0.4856]], "s2-11_0022": [["connector_open", 0.0541, 0.3505, 0.3316, 0.5039]], "s2-11_0023": [["connector_open", 0.071, 0.3507, 0.3451, 0.5047]], "s2-11_0024": [["connector_open", 0.1278, 0.3549, 0.406, 0.5176]], "s2-11_0025": [["connector_open", 0.1998, 0.4184, 0.4793, 0.5688]], "s2-11_0026": [["connector_open", 0.1979, 0.5205, 0.462, 0.6616]], "s2-11_0027": [["connector_open", 0.2485, 0.51, 0.5352, 0.6517]], "s2-11_0028": [["connector_open", 0.3768, 0.618, 0.4716, 0.6776]], "s2-11_0031": [["connector_closed", 0.0002, 0.3986, 0.0881, 0.4876]], "s2-11_0032": [["connector_closed", 0.1506, 0.5142, 0.4071, 0.6339]], "s2-11_0034": [["connector_closed", 0.0014, 0.4591, 0.2009, 0.5743]], "s2-11_0035": [["connector_closed", 0.0016, 0.327, 0.1976, 0.4611]], "s2-11_0036": [["connector_closed", 0.2282, 0.3099, 0.4672, 0.4438]], "s2-11_0037": [["connector_closed", 0.3199, 0.4181, 0.5539, 0.5492]], "s2-11_0038": [["connector_closed", 0.3818, 0.4237, 0.6392, 0.5591]], "s2-11_0039": [["connector_closed", 0.4457, 0.4671, 0.6923, 0.5902]], "s2-11_0040": [["connector_closed", 0.4094, 0.4716, 0.6481, 0.5937]], "s2-11_0041": [["connector_closed", 0.4432, 0.5325, 0.6358, 0.6591]], "s2-11_0043": [["connector_closed", 0.1552, 0.393, 0.4226, 0.54]], "s2-11_0048": [["connector_closed", 0.2936, 0.6343, 0.4112, 0.7228]], "s2-11_0049": [["connector_closed", 0.2721, 0.6485, 0.3656, 0.7024]], "s2-11_0050": [["connector_closed", 0.2346, 0.644, 0.3697, 0.721]], "s2-11_0051": [["connector_closed", 0.2328, 0.6436, 0.3677, 0.7209]], "s2-11_0052": [["connector_closed", 0.2294, 0.6439, 0.3683, 0.7216]], "s2-12_0001": [["connector_closed", 0.4067, 0.5233, 0.5493, 0.6025]], "s2-12_0002": [["connector_closed", 0.4074, 0.5295, 0.5468, 0.6025]], "s2-12_0064": [["connector_closed", 0.3076, 0.3187, 0.5278, 0.4624]], "s2-12_0065": [["connector_closed", 0.3202, 0.3976, 0.538, 0.527]], "s2-12_0066": [["connector_closed", 0.2725, 0.3631, 0.4985, 0.4854]], "s2-12_0067": [["connector_closed", 0.2711, 0.4354, 0.457, 0.5472]], "s2-12_0068": [["connector_closed", 0.2577, 0.4026, 0.4635, 0.5194]], "s2-12_0069": [["connector_closed", 0.2791, 0.372, 0.4887, 0.4936]], "s2-12_0070": [["connector_closed", 0.3348, 0.3892, 0.469, 0.4902]], "s2-12_0071": [["connector_closed", 0.1342, 0.3314, 0.3664, 0.4281]], "s2-12_0072": [["connector_closed", 0.3147, 0.3721, 0.4496, 0.4706]], "s2-12_0073": [["connector_closed", 0.3041, 0.3956, 0.4522, 0.492]], "s2-13_0001": [["connector_closed", 0.2457, 0.6831, 0.4009, 0.773]], "s2-13_0002": [["connector_closed", 0.2501, 0.6835, 0.3956, 0.7725]], "s2-13_0010": [["connector_open", 0.0277, 0.5702, 0.1944, 0.6257]], "s2-13_0011": [["connector_open", 0.0054, 0.5662, 0.1591, 0.6165]], "s2-13_0013": [["connector_open", 0.4617, 0.5521, 0.665, 0.6614]], "s2-13_0014": [["connector_open", 0.5122, 0.5672, 0.7137, 0.6936]], "s2-13_0015": [["connector_open", 0.6003, 0.6007, 0.7629, 0.652]], "s2-13_0016": [["connector_open", 0.5638, 0.6937, 0.7438, 0.7475]], "s2-13_0033": [["connector_closed", 0.3126, 0.5248, 0.5557, 0.6625]], "s2-13_0034": [["connector_closed", 0.3527, 0.5402, 0.5896, 0.6776]], "s2-13_0035": [["connector_closed", 0.4108, 0.5628, 0.6485, 0.6922]], "s2-13_0036": [["connector_closed", 0.4653, 0.5971, 0.7062, 0.7234]], "s2-13_0037": [["connector_closed", 0.5034, 0.6275, 0.7304, 0.7464]], "s2-13_0038": [["connector_closed", 0.5664, 0.6998, 0.8558, 0.9987]], "s2-13_0039": [["connector_closed", 0.5171, 0.7373, 0.7631, 0.8538]], "s2-13_0040": [["connector_closed", 0.1345, 0.3983, 0.3839, 0.5331]], "s2-13_0041": [["connector_closed", 0.2517, 0.4444, 0.5001, 0.5867]], "s2-13_0042": [["connector_closed", 0.3284, 0.6025, 0.535, 0.7246]], "s2-13_0043": [["connector_closed", 0.2638, 0.6609, 0.3437, 0.7299]], "s2-13_0044": [["connector_closed", 0.2109, 0.6434, 0.3274, 0.7224]], "s2-14_0001": [["connector_closed", 0.1992, 0.7041, 0.3483, 0.7913]], "s2-14_0002": [["connector_closed", 0.19, 0.7052, 0.3516, 0.7886]], "s2-14_0003": [["connector_closed", 0.5284, 0.7829, 0.7606, 0.8948]], "s2-14_0004": [["connector_closed", 0.4737, 0.752, 0.6341, 0.8597]], "s2-14_0006": [["connector_closed", 0.3106, 0.7427, 0.5056, 0.8597]], "s2-14_0011": [["connector_open", 0.4846, 0.6224, 0.7167, 0.7351]], "s2-14_0012": [["connector_open", 0.5248, 0.751, 0.7183, 0.8289]], "s2-14_0042": [["connector_closed", 0.0011, 0.5169, 0.0606, 0.5903]], "s2-14_0043": [["connector_closed", 0.0012, 0.5179, 0.0576, 0.5905]], "s2-14_0044": [["connector_closed", 0.0006, 0.5189, 0.0625, 0.5897]], "s2-14_0045": [["connector_closed", 0.0008, 0.52, 0.0594, 0.5908]], "s2-15_0001": [["connector_closed", 0.2622, 0.7145, 0.4041, 0.7947]], "s2-15_0002": [["connector_closed", 0.2665, 0.7172, 0.404, 0.7946]], "s2-15_0003": [["connector_closed", 0.3606, 0.6932, 0.5469, 0.7855]], "s2-15_0004": [["connector_closed", 0.3843, 0.5896, 0.5877, 0.7114]], "s2-15_0005": [["connector_closed", 0.329, 0.6639, 0.5806, 0.784]], "s2-15_0006": [["connector_closed", 0.2738, 0.693, 0.5318, 0.8082]], "s2-15_0007": [["connector_closed", 0.3326, 0.678, 0.5689, 0.8009]], "s2-15_0008": [["connector_closed", 0.0019, 0.6682, 0.2139, 0.7851]], "s2-15_0010": [["connector_open", 0.1536, 0.6681, 0.3417, 0.7791]], "s2-15_0054": [["connector_closed", 0.3198, 0.8313, 0.447, 0.9056]], "s2-15_0055": [["connector_closed", 0.2802, 0.8328, 0.4105, 0.9036]], "s2-15_0056": [["connector_closed", 0.2759, 0.8342, 0.4142, 0.9033]], "s2-15_0057": [["connector_closed", 0.2736, 0.8321, 0.4135, 0.9029]], "s2-16_0001": [["connector_closed", 0.3664, 0.7341, 0.5046, 0.8169]], "s2-16_0002": [["connector_closed", 0.3588, 0.7351, 0.5046, 0.8173]], "s2-16_0003": [["connector_closed", 0.5228, 0.6471, 0.7115, 0.7454]], "s2-16_0004": [["connector_closed", 0.2425, 0.5402, 0.4552, 0.6581]], "s2-16_0020": [["connector_open", 0.2901, 0.4883, 0.3674, 0.5917]], "s2-16_0021": [["connector_open", 0.1011, 0.3596, 0.3289, 0.4978]], "s2-16_0022": [["connector_open", 0.0943, 0.3513, 0.3664, 0.4844]], "s2-16_0023": [["connector_open", 0.1271, 0.3819, 0.4328, 0.5204]], "s2-16_0080": [["connector_closed", 0.2224, 0.6983, 0.3658, 0.784]], "s2-17_0001": [["connector_closed", 0.3974, 0.7824, 0.532, 0.862]], "s2-17_0002": [["connector_closed", 0.3999, 0.7841, 0.533, 0.8598]], "s2-17_0003": [["connector_closed", 0.4975, 0.6657, 0.7154, 0.7754]], "s2-17_0004": [["connector_closed", 0.5068, 0.6945, 0.7032, 0.7964]], "s2-17_0005": [["connector_closed", 0.5488, 0.7408, 0.7373, 0.862]], "s2-17_0015": [["connector_open", 0.2962, 0.7698, 0.3782, 0.8368]], "s2-17_0017": [["connector_open", 0.0018, 0.7585, 0.0538, 0.8223]], "s2-17_0018": [["connector_open", 0.2954, 0.6023, 0.3801, 0.6543]], "s2-17_0019": [["connector_open", 0.3427, 0.6268, 0.5397, 0.7433]], "s2-17_0020": [["connector_open", 0.3648, 0.6465, 0.5889, 0.7833]], "s2-17_0021": [["connector_open", 0.4419, 0.6721, 0.6626, 0.8112]], "s2-17_0022": [["connector_open", 0.5447, 0.7027, 0.6623, 0.7829]], "s2-17_0053": [["connector_open", 0.2291, 0.5992, 0.4671, 0.7185]], "s2-17_0055": [["connector_open", 0.1826, 0.5548, 0.4262, 0.6806]], "s2-17_0056": [["connector_open", 0.0117, 0.5524, 0.268, 0.681]], "s2-17_0057": [["connector_open", 0.3261, 0.5687, 0.4078, 0.666]], "s2-17_0058": [["connector_open", 0.3513, 0.599, 0.4129, 0.6591]], "s2-17_0060": [["connector_open", 0.0, 0.6129, 0.0919, 0.6712]], "s2-17_0088": [["connector_closed", 0.2088, 0.6432, 0.3259, 0.7154]], "s2-17_0089": [["connector_closed", 0.2124, 0.642, 0.3292, 0.7165]], "s2-18_0001": [["connector_closed", 0.3312, 0.7604, 0.4619, 0.8333]], "s2-18_0002": [["connector_closed", 0.3305, 0.7585, 0.4613, 0.83]], "s2-18_0014": [["connector_open", 0.3316, 0.7118, 0.4321, 0.7726]], "s2-18_0019": [["connector_open", 0.3075, 0.5012, 0.535, 0.6357]], "s2-18_0020": [["connector_open", 0.4988, 0.5452, 0.6362, 0.6227]], "s2-18_0021": [["connector_open", 0.4359, 0.5632, 0.5576, 0.6397]], "s2-18_0045": [["connector_closed", 0.2803, 0.5848, 0.3951, 0.6356]], "s2-18_0046": [["connector_closed", 0.2824, 0.5836, 0.3968, 0.6321]], "s2-19_0001": [["connector_closed", 0.5963, 0.5271, 0.6708, 0.6571]], "s2-19_0002": [["connector_closed", 0.5985, 0.5306, 0.6714, 0.6568]], "s2-19_0003": [["connector_closed", 0.5872, 0.5092, 0.6583, 0.6429]], "s2-19_0025": [["connector_open", 0.7867, 0.5114, 0.8663, 0.6584]], "s2-19_0026": [["connector_open", 0.8468, 0.6537, 0.9311, 0.8133]], "s2-19_0027": [["connector_open", 0.8032, 0.5732, 0.8954, 0.7441]], "s2-19_0041": [["connector_closed", 0.4662, 0.2784, 0.6516, 0.5987]], "s2-19_0042": [["connector_closed", 0.2711, 0.5465, 0.4682, 0.9546]], "s2-19_0043": [["connector_closed", 0.5206, 0.2782, 0.8603, 0.8892]], "s2-19_0044": [["connector_closed", 0.3792, 0.2346, 0.7248, 0.8221]], "s2-19_0045": [["connector_closed", 0.4406, 0.3422, 0.744, 0.8204]], "s2-19_0046": [["connector_closed", 0.4525, 0.2943, 0.7205, 0.8933]], "s2-19_0047": [["connector_closed", 0.428, 0.3435, 0.7369, 0.8897]], "s2-19_0048": [["connector_closed", 0.3766, 0.4079, 0.696, 0.9723]], "s2-19_0049": [["connector_closed", 0.4002, 0.3226, 0.7117, 0.8682]], "s2-19_0050": [["connector_closed", 0.4806, 0.238, 0.8045, 0.8084]], "s2-19_0051": [["connector_closed", 0.4605, 0.2855, 0.7992, 0.8436]], "s2-20_0001": [["connector_closed", 0.7809, 0.4989, 0.8707, 0.6577]], "s2-20_0002": [["connector_closed", 0.8001, 0.4975, 0.8718, 0.5992]], "s2-20_0014": [["connector_open", 0.6903, 0.3654, 0.8265, 0.59]], "s2-20_0015": [["connector_open", 0.7045, 0.3602, 0.7835, 0.4595]], "s2-20_0017": [["connector_open", 0.6007, 0.4017, 0.7447, 0.6537]], "s2-20_0025": [["connector_open", 0.5323, 0.1442, 0.7007, 0.4214]], "s2-20_0026": [["connector_open", 0.5887, 0.0867, 0.7512, 0.3597]], "s2-20_0027": [["connector_open", 0.6672, 0.0784, 0.8115, 0.3097]], "s2-20_0057": [["connector_closed", 0.6462, 0.6812, 0.7168, 0.8115]], "s2-20_0058": [["connector_closed", 0.6465, 0.6821, 0.7159, 0.8113]], "s2-20_0059": [["connector_closed", 0.6462, 0.6815, 0.7159, 0.8117]], "s2-21_0001": [["connector_closed", 0.6898, 0.3955, 0.7845, 0.5635]], "s2-21_0002": [["connector_closed", 0.6902, 0.3955, 0.7847, 0.5631]], "s2-21_0013": [["connector_open", 0.3964, 0.349, 0.5843, 0.724]], "s2-21_0014": [["connector_open", 0.4707, 0.3011, 0.6264, 0.614]], "s2-21_0015": [["connector_open", 0.461, 0.2842, 0.6365, 0.624]], "s2-21_0016": [["connector_open", 0.4368, 0.3216, 0.6021, 0.5986]], "s2-21_0017": [["connector_open", 0.4014, 0.3291, 0.6026, 0.6703]], "s2-21_0026": [["connector_open", 0.2968, 0.4413, 0.4954, 0.9115]], "s2-21_0027": [["connector_open", 0.3435, 0.4205, 0.536, 0.8407]], "s2-21_0028": [["connector_open", 0.3172, 0.3141, 0.5055, 0.757]], "s2-21_0029": [["connector_open", 0.3796, 0.5804, 0.5702, 0.8966]], "s2-21_0030": [["connector_open", 0.4214, 0.5451, 0.5971, 0.8637]], "s2-21_0031": [["connector_open", 0.3502, 0.4391, 0.5313, 0.7729]], "s2-21_0032": [["connector_open", 0.3466, 0.4923, 0.5336, 0.8403]], "s2-21_0033": [["connector_open", 0.391, 0.2564, 0.5772, 0.6181]], "s2-21_0034": [["connector_open", 0.366, 0.217, 0.5553, 0.5782]], "s2-21_0035": [["connector_open", 0.3717, 0.0952, 0.5815, 0.4993]], "s2-21_0067": [["connector_closed", 0.6533, 0.6152, 0.7267, 0.7428]], "s2-21_0068": [["connector_closed", 0.6537, 0.6141, 0.7248, 0.7417]], "s2-21_0069": [["connector_closed", 0.654, 0.6132, 0.7258, 0.7413]], "s2-22_0001": [["connector_closed", 0.6422, 0.0381, 0.7252, 0.1891]], "s2-22_0002": [["connector_closed", 0.6427, 0.037, 0.725, 0.1895]], "s2-22_0003": [["connector_closed", 0.6327, 0.0413, 0.7056, 0.1697]], "s2-22_0016": [["connector_open", 0.4349, 0.4238, 0.5554, 0.7045]], "s2-22_0017": [["connector_open", 0.5045, 0.3629, 0.6175, 0.6338]], "s2-22_0018": [["connector_open", 0.5291, 0.2128, 0.6478, 0.5006]], "s2-22_0019": [["connector_open", 0.5369, 0.1606, 0.691, 0.4937]], "s2-22_0020": [["connector_open", 0.5725, 0.2738, 0.7441, 0.5876]], "s2-22_0021": [["connector_open", 0.6575, 0.217, 0.7895, 0.4624]], "s2-22_0050": [["connector_closed", 0.5723, 0.5335, 0.6605, 0.6857]], "s2-22_0051": [["connector_closed", 0.5717, 0.5319, 0.6601, 0.6911]], "s2-23_0001": [["connector_closed", 0.5463, 0.4723, 0.6316, 0.6314]], "s2-23_0002": [["connector_closed", 0.5468, 0.4697, 0.6324, 0.6273]], "s2-23_0016": [["connector_open", 0.5327, 0.183, 0.6682, 0.4299]], "s2-23_0017": [["connector_open", 0.5038, 0.3216, 0.6443, 0.5756]], "s2-23_0018": [["connector_open", 0.5016, 0.2672, 0.654, 0.5485]], "s2-23_0020": [["connector_open", 0.5209, 0.308, 0.6776, 0.5922]], "s2-23_0021": [["connector_open", 0.5771, 0.2345, 0.7321, 0.5028]], "s2-23_0062": [["connector_open", 0.5486, 0.2274, 0.7317, 0.5498]], "s2-23_0089": [["connector_closed", 0.6768, 0.5776, 0.7478, 0.7038]], "s2-23_0090": [["connector_closed", 0.6755, 0.5767, 0.7482, 0.7065]], "s2-24_0061": [["connector_closed", 0.5635, 0.6121, 0.6788, 0.6804]], "s2-24_0062": [["connector_closed", 0.5636, 0.6157, 0.6777, 0.6814]], "s2-24_0063": [["connector_closed", 0.5665, 0.6147, 0.6783, 0.6787]], "s2-24_0064": [["connector_closed", 0.5648, 0.6146, 0.6781, 0.6791]], "s2-24_0065": [["connector_closed", 0.5627, 0.6115, 0.6783, 0.68]], "s2-25_0001": [["connector_closed", 0.4421, 0.4586, 0.505, 0.5705]], "s2-25_0002": [["connector_closed", 0.4417, 0.4576, 0.5062, 0.5689]], "s2-25_0003": [["connector_closed", 0.4417, 0.4569, 0.5071, 0.5699]], "s2-25_0004": [["connector_closed", 0.4417, 0.4572, 0.5063, 0.5693]], "s2-25_0014": [["connector_open", 0.3084, 0.5732, 0.3863, 0.7329]], "s2-25_0015": [["connector_open", 0.1768, 0.3879, 0.2888, 0.5689]], "s2-25_0016": [["connector_open", 0.0876, 0.2863, 0.2147, 0.4897]], "s2-25_0017": [["connector_open", 0.2451, 0.3412, 0.3496, 0.5376]], "s2-25_0018": [["connector_open", 0.2705, 0.4077, 0.4006, 0.6394]], "s2-25_0019": [["connector_open", 0.2764, 0.3801, 0.4039, 0.5869]], "s2-25_0020": [["connector_open", 0.2953, 0.4581, 0.4104, 0.6616]], "s2-25_0021": [["connector_open", 0.2803, 0.5122, 0.3979, 0.7202]], "s2-25_0022": [["connector_open", 0.2609, 0.5339, 0.3925, 0.7285]], "s2-25_0026": [["connector_open", 0.1576, 0.2011, 0.2809, 0.4336]], "s2-25_0027": [["connector_open", 0.2159, 0.2574, 0.3356, 0.4587]], "s2-25_0028": [["connector_open", 0.2361, 0.2833, 0.3482, 0.5114]], "s2-25_0030": [["connector_open", 0.3131, 0.4469, 0.4148, 0.6052]], "s2-25_0031": [["connector_open", 0.3278, 0.3934, 0.4391, 0.6069]], "s2-25_0032": [["connector_open", 0.309, 0.5359, 0.4277, 0.7381]], "s2-25_0033": [["connector_open", 0.3031, 0.5332, 0.4092, 0.7366]], "s2-25_0034": [["connector_open", 0.2735, 0.5363, 0.3825, 0.6878]], "s2-25_0036": [["connector_open", 0.1071, 0.1856, 0.2179, 0.4154]], "s2-25_0037": [["connector_open", 0.1573, 0.3049, 0.2759, 0.5423]], "s2-25_0038": [["connector_open", 0.2055, 0.3221, 0.3173, 0.5636]], "s2-25_0039": [["connector_open", 0.2222, 0.3295, 0.332, 0.5777]], "s2-25_0051": [["connector_closed", 0.4236, 0.1053, 0.4904, 0.2115]], "s2-25_0052": [["connector_closed", 0.4244, 0.1048, 0.4904, 0.2122]], "s2-25_0053": [["connector_closed", 0.4238, 0.1044, 0.4901, 0.2125]], "s2-25_0054": [["connector_closed", 0.4238, 0.1037, 0.4898, 0.2117]], "s2-26_0001": [["connector_closed", 0.2774, 0.2295, 0.3554, 0.3708]], "s2-26_0002": [["connector_closed", 0.2772, 0.2287, 0.3555, 0.3705]], "s2-26_0003": [["connector_closed", 0.2773, 0.2283, 0.3559, 0.3714]], "s2-26_0004": [["connector_closed", 0.2774, 0.2278, 0.3551, 0.3715]], "s2-26_0005": [["connector_closed", 0.2741, 0.2283, 0.356, 0.3738]], "s2-26_0006": [["connector_closed", 0.276, 0.229, 0.3564, 0.373]], "s2-26_0007": [["connector_closed", 0.2772, 0.2282, 0.3551, 0.3723]], "s2-26_0008": [["connector_closed", 0.2755, 0.2248, 0.3567, 0.3744]], "s2-26_0018": [["connector_open", 0.3411, 0.39, 0.3975, 0.469]], "s2-26_0019": [["connector_open", 0.3161, 0.3872, 0.392, 0.5137]], "s2-26_0020": [["connector_open", 0.3158, 0.3925, 0.3994, 0.5245]], "s2-26_0021": [["connector_open", 0.3247, 0.3719, 0.3832, 0.4819]], "s2-26_0022": [["connector_open", 0.2626, 0.3559, 0.3407, 0.4854]], "s2-26_0023": [["connector_open", 0.264, 0.372, 0.3427, 0.5028]], "s2-26_0024": [["connector_open", 0.261, 0.3988, 0.3335, 0.5321]], "s2-26_0025": [["connector_open", 0.233, 0.4034, 0.3124, 0.546]], "s2-26_0026": [["connector_open", 0.2459, 0.4592, 0.3337, 0.5969]], "s2-26_0027": [["connector_open", 0.2274, 0.5204, 0.3047, 0.6349]], "s2-26_0029": [["connector_open", 0.2499, 0.6329, 0.3204, 0.747]], "s2-26_0030": [["connector_open", 0.2647, 0.6032, 0.3353, 0.7262]], "s2-26_0031": [["connector_open", 0.2576, 0.4603, 0.3364, 0.6337]], "s2-26_0032": [["connector_open", 0.2436, 0.4791, 0.324, 0.6278]], "s2-26_0033": [["connector_open", 0.2453, 0.4695, 0.3286, 0.6223]], "s2-26_0034": [["connector_open", 0.2083, 0.5069, 0.2865, 0.6484]], "s2-26_0035": [["connector_open", 0.1781, 0.4943, 0.2648, 0.6488]], "s2-26_0036": [["connector_open", 0.1966, 0.5125, 0.2822, 0.6634]], "s2-26_0037": [["connector_open", 0.1595, 0.4743, 0.259, 0.6257]], "s2-26_0038": [["connector_open", 0.1483, 0.4944, 0.2479, 0.6274]], "s2-26_0048": [["connector_closed", 0.1255, 0.3252, 0.1905, 0.4157]], "s2-26_0049": [["connector_closed", 0.125, 0.3278, 0.1933, 0.4212]], "s2-26_0050": [["connector_closed", 0.167, 0.366, 0.2136, 0.4713]], "s2-26_0051": [["connector_closed", 0.1332, 0.4592, 0.195, 0.5633]], "s2-26_0052": [["connector_closed", 0.1383, 0.3964, 0.1978, 0.4799]], "s2-26_0053": [["connector_closed", 0.1379, 0.3827, 0.1987, 0.4831]], "s2-26_0054": [["connector_closed", 0.1371, 0.3837, 0.2001, 0.4842]], "s2-27_0001": [["connector_closed", 0.1402, 0.3937, 0.2094, 0.517]], "s2-27_0002": [["connector_closed", 0.138, 0.397, 0.2103, 0.5171]], "s2-27_0003": [["connector_closed", 0.1395, 0.3943, 0.2099, 0.5171]], "s2-27_0004": [["connector_closed", 0.1382, 0.3962, 0.2102, 0.5174]], "s2-27_0005": [["connector_closed", 0.1376, 0.3956, 0.2106, 0.5172]], "s2-27_0006": [["connector_closed", 0.1373, 0.3965, 0.2099, 0.5163]], "s2-27_0007": [["connector_closed", 0.1381, 0.3951, 0.2108, 0.5179]], "s2-27_0008": [["connector_closed", 0.1384, 0.3954, 0.2097, 0.5175]], "s2-27_0009": [["connector_closed", 0.1381, 0.3956, 0.2104, 0.5171]], "s2-27_0010": [["connector_closed", 0.1384, 0.3976, 0.2101, 0.5158]], "s2-27_0018": [["connector_open", 0.3119, 0.3484, 0.4085, 0.5402]], "s2-27_0019": [["connector_open", 0.2396, 0.4212, 0.3269, 0.5695]], "s2-27_0020": [["connector_open", 0.0855, 0.444, 0.1937, 0.6069]], "s2-27_0021": [["connector_open", 0.0505, 0.399, 0.163, 0.5786]], "s2-27_0022": [["connector_open", 0.0839, 0.4005, 0.1939, 0.5804]], "s2-27_0023": [["connector_open", 0.1726, 0.4299, 0.2772, 0.6106]], "s2-27_0024": [["connector_open", 0.262, 0.4295, 0.357, 0.5967]], "s2-27_0025": [["connector_open", 0.1396, 0.5632, 0.2396, 0.7372]], "s2-27_0026": [["connector_open", 0.0153, 0.5544, 0.1436, 0.7638]], "s2-27_0027": [["connector_open", 0.0664, 0.4661, 0.1926, 0.6728]], "s2-27_0028": [["connector_open", 0.0854, 0.4548, 0.2073, 0.649]], "s2-27_0029": [["connector_open", 0.1263, 0.4959, 0.2397, 0.6894]], "s2-27_0030": [["connector_open", 0.1713, 0.536, 0.2797, 0.7254]], "s2-27_0031": [["connector_open", 0.2411, 0.5478, 0.3413, 0.7087]], "s2-27_0032": [["connector_open", 0.2911, 0.5617, 0.3833, 0.7201]], "s2-27_0033": [["connector_open", 0.1817, 0.5615, 0.281, 0.7375]], "s2-27_0034": [["connector_open", 0.1207, 0.4896, 0.2309, 0.6687]], "s2-27_0035": [["connector_open", 0.0605, 0.4134, 0.1882, 0.6396]], "s2-27_0036": [["connector_open", 0.1009, 0.4817, 0.2271, 0.6975]], "s2-27_0037": [["connector_open", 0.143, 0.5489, 0.2774, 0.7419]], "s2-27_0038": [["connector_open", 0.1848, 0.5147, 0.3079, 0.7113]], "s2-27_0050": [["connector_closed", 0.2136, 0.3338, 0.2722, 0.4486]], "s2-27_0051": [["connector_closed", 0.2132, 0.3349, 0.2746, 0.4481]], "s2-27_0052": [["connector_closed", 0.2131, 0.3359, 0.2756, 0.4486]], "s2-28_0009": [["connector_open", 0.3574, 0.007, 0.4316, 0.1321]], "s2-28_0010": [["connector_open", 0.356, 0.0057, 0.4322, 0.1324]], "s2-28_0011": [["connector_open", 0.3538, 0.0031, 0.44, 0.1347]], "s2-28_0012": [["connector_open", 0.3573, 0.0038, 0.4342, 0.1307]], "s2-28_0013": [["connector_open", 0.3564, 0.0032, 0.4347, 0.1316]], "s2-28_0014": [["connector_open", 0.3567, 0.0029, 0.4379, 0.1311]], "s2-28_0015": [["connector_open", 0.3572, 0.0065, 0.4344, 0.1365]], "s2-28_0016": [["connector_open", 0.3558, 0.007, 0.4352, 0.1371]], "s2-28_0017": [["connector_open", 0.3555, 0.0051, 0.4335, 0.1369]], "s2-28_0018": [["connector_open", 0.3533, 0.0064, 0.4371, 0.1401]], "s2-28_0023": [["connector_open", 0.0816, 0.6064, 0.2044, 0.7808]], "s2-28_0024": [["connector_open", 0.2846, 0.3963, 0.3888, 0.5721]], "s2-28_0025": [["connector_open", 0.3403, 0.4057, 0.4298, 0.572]], "s2-28_0026": [["connector_open", 0.3248, 0.2762, 0.4137, 0.4645]], "s2-28_0027": [["connector_open", 0.2952, 0.332, 0.3832, 0.4851]], "s2-28_0030": [["connector_open", 0.1681, 0.3045, 0.2779, 0.4852]], "s2-28_0031": [["connector_open", 0.1791, 0.3254, 0.2923, 0.5057]], "s2-28_0032": [["connector_open", 0.1697, 0.3336, 0.2883, 0.5332]], "s2-28_0033": [["connector_open", 0.2308, 0.307, 0.363, 0.5029]], "s2-28_0034": [["connector_open", 0.2516, 0.3417, 0.3722, 0.5305]], "s2-28_0035": [["connector_open", 0.0003, 0.4717, 0.0998, 0.6555]], "s2-28_0036": [["connector_open", 0.0395, 0.3616, 0.1572, 0.5403]], "s2-28_0037": [["connector_open", 0.1244, 0.3961, 0.2395, 0.589]], "s2-28_0038": [["connector_open", 0.1626, 0.4034, 0.2866, 0.6086]], "s2-28_0039": [["connector_open", 0.1695, 0.425, 0.2932, 0.6423]], "s2-28_0040": [["connector_open", 0.236, 0.3369, 0.3495, 0.5299]], "s2-28_0041": [["connector_open", 0.2857, 0.1718, 0.3792, 0.3604]], "s2-28_0051": [["connector_closed", 0.169, 0.3045, 0.2243, 0.4185]], "s2-28_0052": [["connector_closed", 0.1693, 0.3043, 0.2244, 0.4167]], "s2-28_0053": [["connector_closed", 0.1691, 0.3047, 0.2252, 0.4171]], "s2-29_0001": [["connector_closed", 0.3098, 0.1061, 0.3727, 0.224]], "s2-29_0002": [["connector_closed", 0.3078, 0.1094, 0.3695, 0.2272]], "s2-29_0003": [["connector_closed", 0.3081, 0.1095, 0.3736, 0.225]], "s2-29_0004": [["connector_closed", 0.3087, 0.1096, 0.3709, 0.2241]], "s2-29_0005": [["connector_closed", 0.3083, 0.1067, 0.3724, 0.2251]], "s2-29_0006": [["connector_closed", 0.3093, 0.1097, 0.3728, 0.2232]], "s2-29_0007": [["connector_closed", 0.3086, 0.1086, 0.3749, 0.226]], "s2-29_0008": [["connector_closed", 0.3076, 0.1066, 0.3696, 0.223]], "s2-29_0009": [["connector_closed", 0.309, 0.1069, 0.3724, 0.2233]], "s2-29_0010": [["connector_closed", 0.3159, 0.1134, 0.3687, 0.2236]], "s2-29_0018": [["connector_open", 0.3688, 0.3848, 0.4437, 0.5203]], "s2-29_0019": [["connector_open", 0.203, 0.4006, 0.2885, 0.5477]], "s2-29_0020": [["connector_open", 0.0871, 0.3398, 0.1986, 0.5082]], "s2-29_0021": [["connector_open", 0.2062, 0.3847, 0.3172, 0.5606]], "s2-29_0022": [["connector_open", 0.2247, 0.4433, 0.3313, 0.6244]], "s2-29_0023": [["connector_open", 0.2476, 0.4751, 0.3528, 0.6579]], "s2-29_0024": [["connector_open", 0.2607, 0.5208, 0.3634, 0.6875]], "s2-29_0025": [["connector_open", 0.3635, 0.5226, 0.4476, 0.6674]], "s2-29_0026": [["connector_open", 0.2863, 0.6002, 0.3657, 0.7217]], "s2-29_0028": [["connector_open", 0.147, 0.5284, 0.2396, 0.7063]], "s2-29_0029": [["connector_open", 0.2008, 0.3779, 0.2828, 0.5343]], "s2-29_0030": [["connector_open", 0.2374, 0.4155, 0.3282, 0.5964]], "s2-29_0031": [["connector_open", 0.2489, 0.4442, 0.3433, 0.6308]], "s2-29_0032": [["connector_open", 0.2782, 0.4537, 0.3719, 0.6336]], "s2-29_0033": [["connector_open", 0.2802, 0.5637, 0.3717, 0.7262]], "s2-29_0034": [["connector_open", 0.2845, 0.5811, 0.3868, 0.7416]], "s2-29_0035": [["connector_open", 0.2666, 0.6057, 0.3761, 0.7937]], "s2-29_0036": [["connector_open", 0.2548, 0.655, 0.3425, 0.7615]], "s2-29_0037": [["connector_open", 0.1958, 0.5731, 0.288, 0.7399]], "s2-29_0038": [["connector_open", 0.2574, 0.2475, 0.3512, 0.4285]], "s2-29_0039": [["connector_open", 0.2925, 0.2368, 0.3784, 0.4188]], "s2-29_0040": [["connector_open", 0.2931, 0.2756, 0.3892, 0.4638]], "s2-29_0041": [["connector_open", 0.288, 0.3391, 0.3802, 0.4906]], "s2-29_0050": [["connector_closed", 0.245, 0.3083, 0.3024, 0.4266]], "s2-29_0051": [["connector_closed", 0.2453, 0.3117, 0.3032, 0.425]], "s2-29_0052": [["connector_closed", 0.2462, 0.3113, 0.3026, 0.4235]], "s2-30_0001": [["connector_closed", 0.1968, 0.3928, 0.2603, 0.5026]], "s2-30_0002": [["connector_closed", 0.1971, 0.3922, 0.2594, 0.5049]], "s2-30_0003": [["connector_closed", 0.1971, 0.392, 0.2601, 0.5057]], "s2-30_0004": [["connector_closed", 0.1966, 0.3913, 0.2594, 0.5044]], "s2-30_0005": [["connector_closed", 0.1962, 0.3892, 0.2582, 0.5003]], "s2-30_0006": [["connector_closed", 0.1966, 0.3873, 0.2629, 0.5021]], "s2-30_0007": [["connector_closed", 0.197, 0.3906, 0.2607, 0.5067]], "s2-30_0008": [["connector_closed", 0.196, 0.3842, 0.26, 0.502]], "s2-30_0018": [["connector_open", 0.2916, 0.5076, 0.3772, 0.6817]], "s2-30_0019": [["connector_open", 0.1646, 0.482, 0.2412, 0.6203]], "s2-30_0020": [["connector_open", 0.2074, 0.4843, 0.2853, 0.632]], "s2-30_0021": [["connector_open", 0.2197, 0.473, 0.2933, 0.61]], "s2-30_0022": [["connector_open", 0.2225, 0.4807, 0.3036, 0.6253]], "s2-30_0023": [["connector_open", 0.2445, 0.5014, 0.3256, 0.636]], "s2-30_0024": [["connector_open", 0.2605, 0.5503, 0.3436, 0.6892]], "s2-30_0025": [["connector_open", 0.2728, 0.5667, 0.3555, 0.7044]], "s2-30_0026": [["connector_open", 0.2572, 0.608, 0.3473, 0.7549]], "s2-30_0027": [["connector_open", 0.2852, 0.609, 0.3539, 0.7098]], "s2-30_0028": [["connector_open", 0.2206, 0.5804, 0.3008, 0.7215]], "s2-30_0029": [["connector_open", 0.1604, 0.5406, 0.2472, 0.6777]], "s2-30_0030": [["connector_open", 0.1823, 0.5882, 0.2557, 0.7272]], "s2-30_0031": [["connector_open", 0.1765, 0.5628, 0.2538, 0.707]], "s2-30_0032": [["connector_open", 0.1804, 0.5771, 0.2602, 0.7164]], "s2-30_0033": [["connector_open", 0.2051, 0.6032, 0.2934, 0.736]], "s2-30_0034": [["connector_open", 0.2144, 0.5721, 0.2932, 0.7195]], "s2-30_0035": [["connector_open", 0.2135, 0.672, 0.2923, 0.817]], "s2-30_0050": [["connector_closed", 0.2175, 0.42, 0.2778, 0.5204]], "s2-30_0051": [["connector_closed", 0.2132, 0.424, 0.2786, 0.5207]], "s2-30_0052": [["connector_closed", 0.2157, 0.4234, 0.2791, 0.5188]], "s2-30_0053": [["connector_closed", 0.2248, 0.4523, 0.2977, 0.5636]], "s2-30_0054": [["connector_closed", 0.2206, 0.4429, 0.2856, 0.557]], "s2-30_0055": [["connector_closed", 0.2381, 0.4175, 0.284, 0.5179]], "s2-30_0056": [["connector_closed", 0.2518, 0.4054, 0.2819, 0.4993]], "s2-30_0057": [["connector_closed", 0.2463, 0.3984, 0.2785, 0.4969]], "s2-30_0058": [["connector_closed", 0.2536, 0.3941, 0.283, 0.494]], "s2-30_0059": [["connector_closed", 0.228, 0.3954, 0.2879, 0.5017]], "s2-30_0060": [["connector_closed", 0.2385, 0.4074, 0.2966, 0.5113]], "s2-30_0061": [["connector_closed", 0.2308, 0.4048, 0.2947, 0.5135]], "s2-30_0062": [["connector_closed", 0.2265, 0.3989, 0.292, 0.51]], "s2-30_0063": [["connector_closed", 0.2347, 0.3783, 0.3002, 0.4947]], "s2-30_0064": [["connector_closed", 0.2347, 0.3773, 0.3021, 0.5029]], "s2-31_0041": [["connector_open", 0.0009, 0.3831, 0.1079, 0.6043]], "s2-31_0042": [["connector_open", 0.0007, 0.4253, 0.1274, 0.661]], "s2-31_0043": [["connector_open", 0.0428, 0.4398, 0.1957, 0.6902]], "s2-31_0044": [["connector_open", 0.0004, 0.357, 0.1018, 0.5752]], "s2-31_0060": [["connector_closed", 0.3238, 0.2084, 0.3942, 0.3295]], "s2-31_0061": [["connector_closed", 0.3252, 0.2103, 0.3941, 0.3294]], "s2-31_0062": [["connector_closed", 0.3261, 0.212, 0.3935, 0.3297]], "s2-31_0063": [["connector_closed", 0.3257, 0.2108, 0.3933, 0.3305]], "s2-32_0001": [["connector_closed", 0.256, 0.2112, 0.3437, 0.3632]], "s2-32_0002": [["connector_closed", 0.2538, 0.2048, 0.3386, 0.3602]], "s2-32_0003": [["connector_closed", 0.254, 0.2088, 0.3415, 0.3654]], "s2-32_0004": [["connector_closed", 0.2551, 0.2156, 0.3404, 0.3653]], "s2-32_0005": [["connector_closed", 0.2549, 0.2157, 0.3423, 0.3679]], "s2-32_0006": [["connector_closed", 0.2547, 0.2115, 0.3411, 0.3638]], "s2-32_0007": [["connector_closed", 0.2543, 0.2103, 0.3422, 0.364]], "s2-32_0008": [["connector_closed", 0.255, 0.2076, 0.3358, 0.36]], "s2-32_0009": [["connector_closed", 0.2543, 0.2108, 0.335, 0.3642]], "s2-32_0012": [["connector_open", 0.3199, 0.4331, 0.3826, 0.5917]], "s2-32_0059": [["connector_closed", 0.3272, 0.474, 0.3961, 0.5828]], "s2-32_0060": [["connector_closed", 0.3134, 0.4326, 0.3687, 0.5298]], "s2-32_0061": [["connector_closed", 0.3083, 0.4088, 0.371, 0.533]], "s2-32_0062": [["connector_closed", 0.3085, 0.4117, 0.3704, 0.5353]], "s2-33_0016": [["connector_open", 0.2938, 0.4057, 0.361, 0.5225]], "s2-33_0017": [["connector_open", 0.2958, 0.4044, 0.3606, 0.5218]], "s2-33_0018": [["connector_open", 0.2941, 0.4027, 0.3613, 0.5229]], "s2-33_0019": [["connector_open", 0.2955, 0.4023, 0.3614, 0.5217]], "s2-33_0020": [["connector_open", 0.2954, 0.4033, 0.361, 0.5221]], "s2-33_0021": [["connector_open", 0.2943, 0.403, 0.3604, 0.5235]], "s2-33_0022": [["connector_open", 0.8014, 0.3583, 0.8269, 0.4086]], "s2-33_0023": [["connector_open", 0.2948, 0.4014, 0.3605, 0.5203]], "s2-33_0024": [["connector_open", 0.2959, 0.4058, 0.3625, 0.5263]], "s2-33_0025": [["connector_open", 0.2927, 0.4264, 0.3597, 0.5371]], "s2-33_0053": [["connector_closed", 0.2504, 0.3894, 0.3241, 0.5358]], "s2-33_0054": [["connector_closed", 0.2503, 0.3907, 0.3239, 0.5373]], "s2-33_0055": [["connector_closed", 0.2505, 0.3867, 0.3234, 0.5338]], "s2-34_0001": [["connector_closed", 0.3417, 0.9154, 0.3984, 1.0]], "s2-34_0002": [["connector_closed", 0.3412, 0.9161, 0.3984, 1.0]], "s2-34_0003": [["connector_closed", 0.3419, 0.9172, 0.3977, 1.0]], "s2-34_0004": [["connector_closed", 0.3398, 0.9091, 0.3992, 1.0]], "s2-34_0057": [["connector_closed", 0.4801, 0.1324, 0.5401, 0.2311]], "s2-34_0058": [["connector_closed", 0.4909, 0.1268, 0.5405, 0.216]], "s2-35_0003": [["connector_closed", 0.3402, 0.6481, 0.3961, 0.7618]], "s2-35_0004": [["connector_closed", 0.4157, 0.4721, 0.4801, 0.5923]], "s2-35_0043": [["connector_closed", 0.2628, 0.4096, 0.3237, 0.5132]], "s2-35_0044": [["connector_closed", 0.2832, 0.543, 0.318, 0.6395]], "s2-35_0052": [["connector_closed", 0.237, 0.5029, 0.2858, 0.5815]], "s2-35_0053": [["connector_closed", 0.234, 0.4964, 0.2829, 0.5901]], "s2-35_0054": [["connector_closed", 0.2329, 0.4901, 0.2797, 0.5806]], "s2-35_0055": [["connector_closed", 0.233, 0.4601, 0.285, 0.5517]], "s2-35_0056": [["connector_closed", 0.2329, 0.457, 0.2852, 0.5515]], "s2-36_0001": [["connector_closed", 0.3286, 0.3287, 0.3798, 0.4236]], "s2-36_0002": [["connector_closed", 0.3283, 0.3307, 0.3784, 0.4246]], "s2-36_0003": [["connector_closed", 0.3285, 0.3312, 0.3792, 0.4268]], "s2-36_0004": [["connector_closed", 0.3293, 0.3309, 0.3782, 0.4259]], "s2-36_0005": [["connector_closed", 0.3289, 0.3317, 0.3786, 0.427]], "s2-36_0006": [["connector_closed", 0.3291, 0.3335, 0.3791, 0.427]], "s2-36_0007": [["connector_closed", 0.3288, 0.331, 0.3782, 0.4261]], "s2-36_0008": [["connector_closed", 0.3286, 0.331, 0.3792, 0.4267]], "s2-36_0009": [["connector_closed", 0.3294, 0.333, 0.3792, 0.4273]], "s2-36_0010": [["connector_closed", 0.3289, 0.331, 0.3782, 0.4259]], "s2-36_0011": [["connector_closed", 0.3289, 0.3299, 0.3788, 0.4245]], "s2-36_0056": [["connector_closed", 0.313, 0.4285, 0.3711, 0.5392]], "s2-37_0001": [["connector_closed", 0.2844, 0.4319, 0.3468, 0.5451]], "s2-37_0002": [["connector_closed", 0.2841, 0.4315, 0.3469, 0.5444]], "s2-37_0003": [["connector_closed", 0.2826, 0.4314, 0.3448, 0.5336]], "s2-37_0004": [["connector_closed", 0.2847, 0.4314, 0.3465, 0.5449]], "s2-37_0005": [["connector_closed", 0.282, 0.4298, 0.3469, 0.5449]], "s2-37_0006": [["connector_closed", 0.2964, 0.4362, 0.332, 0.4841]], "s2-37_0007": [["connector_closed", 0.2832, 0.4323, 0.3452, 0.5445]], "s2-37_0008": [["connector_closed", 0.2845, 0.43, 0.3455, 0.5432]], "s2-37_0016": [["connector_open", 0.124, 0.538, 0.1954, 0.6572]], "s2-37_0017": [["connector_open", 0.063, 0.6299, 0.1336, 0.7603]], "s2-37_0041": [["connector_closed", 0.2902, 0.488, 0.3441, 0.5817]], "s2-37_0042": [["connector_closed", 0.2785, 0.5179, 0.3284, 0.6107]], "s2-37_0043": [["connector_closed", 0.2802, 0.5196, 0.3296, 0.6119]], "s2-37_0044": [["connector_closed", 0.3277, 0.5678, 0.6641, 0.8109]], "s2-37_0045": [["connector_closed", 0.28, 0.532, 0.328, 0.6078]], "s2-37_0046": [["connector_closed", 0.2885, 0.5273, 0.3379, 0.6171]], "s2-37_0047": [["connector_closed", 0.2834, 0.5213, 0.3369, 0.6261]], "s2-37_0048": [["connector_closed", 0.2834, 0.521, 0.3368, 0.6269]], "s2-37_0049": [["connector_closed", 0.2879, 0.5224, 0.3388, 0.6269]], "s2-38_0001": [["connector_closed", 0.2496, 0.4461, 0.3027, 0.5444]], "s2-38_0002": [["connector_closed", 0.25, 0.447, 0.3022, 0.5439]], "s2-38_0003": [["connector_closed", 0.2517, 0.4454, 0.303, 0.5439]], "s2-38_0004": [["connector_closed", 0.2507, 0.4465, 0.3028, 0.544]], "s2-38_0005": [["connector_closed", 0.251, 0.4458, 0.3027, 0.5434]], "s2-38_0006": [["connector_closed", 0.2509, 0.4464, 0.3031, 0.5444]], "s2-38_0007": [["connector_closed", 0.2515, 0.4465, 0.3026, 0.5431]], "s2-38_0018": [["connector_open", 0.2726, 0.4929, 0.3268, 0.6009]], "s2-38_0019": [["connector_open", 0.2143, 0.5497, 0.2924, 0.6782]], "s2-38_0020": [["connector_open", 0.2254, 0.7116, 0.3083, 0.8566]], "s2-38_0021": [["connector_open", 0.2315, 0.6642, 0.3191, 0.8253]], "s2-38_0042": [["connector_closed", 0.3998, 0.3107, 0.4557, 0.4158]], "s2-38_0043": [["connector_closed", 0.399, 0.3066, 0.4568, 0.4079]], "s2-38_0044": [["connector_closed", 0.3994, 0.3075, 0.4574, 0.4087]], "s2-38_0045": [["connector_closed", 0.3994, 0.307, 0.458, 0.4086]], "s2-38_0046": [["connector_closed", 0.3999, 0.3054, 0.4564, 0.4046]], "s2-39_0001": [["connector_closed", 0.345, 0.7749, 0.3819, 0.9106]], "s2-39_0002": [["connector_closed", 0.2812, 0.2803, 0.3451, 0.3951]], "s2-39_0003": [["connector_closed", 0.2814, 0.2817, 0.3453, 0.3955]], "s2-39_0004": [["connector_closed", 0.2821, 0.2791, 0.3451, 0.3948]], "s2-39_0005": [["connector_closed", 0.2816, 0.2829, 0.3444, 0.4001]], "s2-39_0006": [["connector_closed", 0.2817, 0.2828, 0.3449, 0.3988]], "s2-39_0007": [["connector_closed", 0.2806, 0.2836, 0.3455, 0.4028]], "s2-39_0023": [["connector_open", 0.1552, 0.4049, 0.2259, 0.5263]], "s2-39_0024": [["connector_open", 0.1721, 0.4918, 0.2455, 0.6212]], "s2-39_0025": [["connector_open", 0.1619, 0.4926, 0.233, 0.6197]], "s2-39_0026": [["connector_open", 0.1596, 0.5134, 0.2312, 0.637]], "s2-39_0038": [["connector_closed", 0.4291, 0.2733, 0.481, 0.3671]], "s2-39_0039": [["connector_closed", 0.4233, 0.2873, 0.4772, 0.3818]], "s2-39_0040": [["connector_closed", 0.4136, 0.2758, 0.4683, 0.3693]], "s2-39_0041": [["connector_closed", 0.4175, 0.2934, 0.4712, 0.3877]], "s2-39_0052": [["connector_closed", 0.2544, 0.5125, 0.3066, 0.6154]], "s2-39_0053": [["connector_closed", 0.2555, 0.5156, 0.3059, 0.616]], "s2-39_0054": [["connector_closed", 0.2636, 0.4944, 0.3155, 0.5914]], "s2-39_0055": [["connector_closed", 0.2651, 0.4941, 0.3155, 0.5902]], "s2-40_0001": [["connector_closed", 0.318, 0.402, 0.3829, 0.5164]], "s2-40_0002": [["connector_closed", 0.3185, 0.4013, 0.3834, 0.5149]], "s2-40_0003": [["connector_closed", 0.319, 0.3996, 0.3852, 0.5171]], "s2-40_0015": [["connector_open", 0.2995, 0.5366, 0.3775, 0.6749]], "s2-40_0016": [["connector_open", 0.2875, 0.5836, 0.373, 0.7305]], "s2-40_0017": [["connector_open", 0.3178, 0.5729, 0.401, 0.7099]], "s2-40_0018": [["connector_open", 0.333, 0.5794, 0.3941, 0.7024]], "s2-40_0036": [["connector_closed", 0.4195, 0.3037, 0.4797, 0.4035]], "s2-40_0037": [["connector_closed", 0.4212, 0.3038, 0.4791, 0.4031]], "s2-40_0038": [["connector_closed", 0.4205, 0.3025, 0.4801, 0.3995]], "s2-41_0006": [["connector_open", 0.1999, 0.5043, 0.2592, 0.6082]], "s2-41_0008": [["connector_open", 0.2154, 0.569, 0.2707, 0.6699]], "s2-41_0009": [["connector_open", 0.2165, 0.566, 0.2712, 0.6627]], "s2-41_0010": [["connector_open", 0.204, 0.624, 0.2629, 0.7316]], "s2-41_0011": [["connector_open", 0.2338, 0.6625, 0.3067, 0.7838]], "s2-41_0012": [["connector_open", 0.234, 0.6549, 0.3037, 0.7865]], "s2-41_0013": [["connector_open", 0.2445, 0.688, 0.3048, 0.808]], "s2-41_0036": [["connector_closed", 0.2237, 0.3976, 0.2803, 0.5061]], "s2-41_0037": [["connector_closed", 0.2218, 0.3979, 0.2802, 0.5079]], "s2-42_0001": [["connector_closed", 0.2486, 0.4961, 0.3008, 0.5942]], "s2-42_0002": [["connector_closed", 0.2487, 0.4965, 0.3004, 0.5918]], "s2-42_0003": [["connector_closed", 0.2483, 0.4967, 0.3006, 0.5926]], "s2-42_0004": [["connector_closed", 0.2496, 0.4958, 0.3004, 0.5912]], "s2-42_0005": [["connector_closed", 0.2445, 0.4945, 0.2988, 0.5895]], "s2-42_0006": [["connector_closed", 0.2526, 0.4973, 0.2997, 0.589]], "s2-42_0007": [["connector_closed", 0.2493, 0.4951, 0.2998, 0.5938]], "s2-42_0008": [["connector_closed", 0.2501, 0.4961, 0.3005, 0.5918]], "s2-42_0020": [["connector_open", 0.2533, 0.6382, 0.3015, 0.7254]], "s2-42_0021": [["connector_open", 0.2533, 0.6391, 0.3093, 0.7421]], "s2-42_0022": [["connector_open", 0.2686, 0.6588, 0.321, 0.7594]], "s2-42_0023": [["connector_open", 0.262, 0.6547, 0.3207, 0.7585]], "s2-42_0024": [["connector_open", 0.2601, 0.6533, 0.3088, 0.738]], "s2-42_0039": [["connector_closed", 0.2494, 0.4673, 0.301, 0.5573]], "s2-42_0040": [["connector_closed", 0.2495, 0.4663, 0.302, 0.56]], "s2-42_0041": [["connector_closed", 0.2522, 0.4679, 0.3023, 0.5626]], "s2-42_0042": [["connector_closed", 0.2498, 0.4678, 0.3029, 0.5633]], "s2-42_0043": [["connector_closed", 0.2505, 0.4668, 0.302, 0.5611]], "s2-43_0001": [["connector_closed", 0.1777, 0.4912, 0.2285, 0.5884]], "s2-43_0002": [["connector_closed", 0.1771, 0.4927, 0.2287, 0.5878]], "s2-43_0003": [["connector_closed", 0.1776, 0.4949, 0.2278, 0.5866]], "s2-43_0004": [["connector_closed", 0.1795, 0.4951, 0.2277, 0.5868]], "s2-43_0005": [["connector_closed", 0.1819, 0.494, 0.2282, 0.5872]], "s2-43_0018": [["connector_open", 0.1798, 0.5967, 0.2426, 0.7064]], "s2-43_0019": [["connector_open", 0.2136, 0.6335, 0.2709, 0.749]], "s2-43_0020": [["connector_open", 0.2094, 0.6402, 0.2678, 0.7343]], "s2-43_0021": [["connector_open", 0.1889, 0.6372, 0.2414, 0.7334]], "s2-43_0022": [["connector_open", 0.1802, 0.603, 0.2368, 0.7077]], "s2-43_0023": [["connector_open", 0.167, 0.6123, 0.2225, 0.7162]], "s2-43_0024": [["connector_open", 0.1785, 0.6295, 0.2397, 0.7331]], "s2-43_0038": [["connector_closed", 0.1341, 0.5065, 0.2036, 0.6293]], "s2-43_0039": [["connector_closed", 0.1304, 0.5206, 0.2058, 0.6447]], "s2-43_0040": [["connector_closed", 0.2176, 0.4751, 0.2619, 0.5619]], "s2-43_0041": [["connector_closed", 0.2423, 0.3858, 0.2794, 0.4681]], "s2-43_0042": [["connector_closed", 0.2438, 0.3792, 0.2852, 0.4668]], "s2-44_0001": [["connector_closed", 0.2897, 0.8453, 0.3462, 0.9932]], "s2-44_0002": [["connector_closed", 0.1816, 0.3728, 0.2266, 0.4631]], "s2-44_0003": [["connector_closed", 0.1818, 0.3766, 0.2267, 0.4659]], "s2-44_0004": [["connector_closed", 0.1816, 0.376, 0.2266, 0.4655]], "s2-44_0005": [["connector_closed", 0.1819, 0.3765, 0.2261, 0.4657]], "s2-44_0006": [["connector_closed", 0.1825, 0.3779, 0.2261, 0.4656]], "s2-44_0007": [["connector_closed", 0.1818, 0.3771, 0.2264, 0.4659]], "s2-44_0008": [["connector_closed", 0.1831, 0.3782, 0.2266, 0.4652]], "s2-44_0009": [["connector_closed", 0.1827, 0.3764, 0.2265, 0.4662]], "s2-44_0051": [["connector_closed", 0.2876, 0.1987, 0.3359, 0.2958]], "s2-45_0004": [["connector_closed", 0.0945, 0.313, 0.1764, 0.4727]], "s2-45_0005": [["connector_closed", 0.0944, 0.3146, 0.1764, 0.4744]], "s2-45_0006": [["connector_closed", 0.0935, 0.3109, 0.1773, 0.4751]], "s2-45_0007": [["connector_closed", 0.0952, 0.3104, 0.1767, 0.472]], "s2-45_0008": [["connector_closed", 0.0945, 0.316, 0.1773, 0.474]], "s2-45_0015": [["connector_open", 0.0009, 0.5413, 0.0951, 0.7424]], "s2-45_0016": [["connector_open", 0.0018, 0.5505, 0.047, 0.7679]], "s2-45_0018": [["connector_open", 0.0, 0.6834, 0.0455, 0.8806]], "s2-45_0020": [["connector_open", 0.0008, 0.4905, 0.0652, 0.6708]], "s2-45_0021": [["connector_open", 0.0004, 0.4791, 0.0698, 0.731]], "s2-45_0024": [["connector_open", 0.203, 0.3314, 0.3373, 0.5274]], "s2-45_0025": [["connector_open", 0.0007, 0.0062, 0.1152, 0.2432]], "s2-45_0026": [["connector_open", 0.0003, 0.0763, 0.0947, 0.2925]], "s2-45_0028": [["connector_open", 0.0013, 0.0251, 0.0925, 0.2701]], "s2-45_0029": [["connector_open", 0.0004, 0.0044, 0.0547, 0.2653]], "s2-45_0030": [["connector_open", 0.0008, 0.1093, 0.1196, 0.3584]], "s2-45_0031": [["connector_open", 0.0955, 0.2135, 0.2281, 0.4411]], "s2-45_0032": [["connector_open", 0.0008, 0.2236, 0.0736, 0.4361]], "s2-45_0033": [["connector_open", 0.2391, 0.0015, 0.3814, 0.2204]], "s2-45_0034": [["connector_open", 0.3566, 0.1237, 0.4692, 0.3108]], "s2-45_0035": [["connector_open", 0.3381, 0.2537, 0.4374, 0.4225]], "s2-45_0036": [["connector_open", 0.3342, 0.2422, 0.4333, 0.4157]], "s2-45_0037": [["connector_open", 0.3353, 0.2526, 0.4278, 0.4225]], "s2-45_0049": [["connector_closed", 0.1306, 0.2588, 0.2198, 0.4322]], "s2-45_0050": [["connector_closed", 0.1178, 0.2582, 0.2187, 0.4347]], "s2-45_0051": [["connector_closed", 0.1173, 0.2589, 0.2172, 0.434]], "s2-46_0001": [["connector_closed", 0.0007, 0.2983, 0.0817, 0.4558]], "s2-46_0002": [["connector_closed", 0.0008, 0.2997, 0.0824, 0.454]], "s2-46_0003": [["connector_closed", 0.0152, 0.2902, 0.0943, 0.4308]], "s2-46_0012": [["connector_open", 0.1625, 0.3786, 0.2526, 0.5414]], "s2-46_0013": [["connector_open", 0.0378, 0.331, 0.1393, 0.4993]], "s2-46_0014": [["connector_open", 0.0828, 0.327, 0.1886, 0.5036]], "s2-46_0015": [["connector_open", 0.1553, 0.3603, 0.2516, 0.5257]], "s2-46_0016": [["connector_open", 0.1531, 0.3547, 0.2459, 0.5185]], "s2-46_0017": [["connector_open", 0.1626, 0.4232, 0.2596, 0.5817]], "s2-46_0018": [["connector_open", 0.1794, 0.4306, 0.2788, 0.596]], "s2-46_0019": [["connector_open", 0.1689, 0.4296, 0.2702, 0.5997]], "s2-46_0020": [["connector_open", 0.1515, 0.3884, 0.2496, 0.5591]], "s2-46_0021": [["connector_open", 0.1571, 0.3891, 0.2595, 0.5566]], "s2-46_0022": [["connector_open", 0.1631, 0.3922, 0.2551, 0.5559]], "s2-46_0023": [["connector_open", 0.1728, 0.4271, 0.2656, 0.5916]], "s2-46_0024": [["connector_open", 0.1809, 0.4315, 0.2764, 0.5953]], "s2-46_0025": [["connector_open", 0.2161, 0.354, 0.3144, 0.5145]], "s2-46_0026": [["connector_open", 0.2383, 0.3256, 0.335, 0.4804]], "s2-46_0027": [["connector_open", 0.2326, 0.3228, 0.3362, 0.4832]], "s2-46_0028": [["connector_open", 0.2175, 0.3374, 0.3191, 0.501]], "s2-46_0029": [["connector_open", 0.2327, 0.3886, 0.3296, 0.5463]], "s2-46_0030": [["connector_open", 0.1827, 0.2604, 0.2968, 0.4642]], "s2-46_0031": [["connector_open", 0.2029, 0.1968, 0.3061, 0.4071]], "s2-46_0042": [["connector_closed", 0.0404, 0.4167, 0.1201, 0.5682]], "s2-46_0043": [["connector_closed", 0.0403, 0.4241, 0.1232, 0.5691]], "s2-46_0044": [["connector_closed", 0.0559, 0.5451, 0.1539, 0.6639]], "s2-46_0045": [["connector_closed", 0.0369, 0.3878, 0.1299, 0.5672]], "s2-46_0046": [["connector_closed", 0.0556, 0.3261, 0.1478, 0.4844]], "s2-46_0047": [["connector_closed", 0.058, 0.3007, 0.1451, 0.4571]], "s2-47_0001": [["connector_closed", 0.0275, 0.4025, 0.1225, 0.5689]], "s2-47_0002": [["connector_closed", 0.0286, 0.4044, 0.1236, 0.5718]], "s2-47_0003": [["connector_closed", 0.0274, 0.4037, 0.1209, 0.5685]], "s2-47_0004": [["connector_closed", 0.0278, 0.4054, 0.1221, 0.57]], "s2-47_0005": [["connector_closed", 0.0266, 0.4026, 0.1209, 0.5676]], "s2-47_0006": [["connector_closed", 0.0289, 0.4021, 0.122, 0.5674]], "s2-47_0007": [["connector_closed", 0.0275, 0.4025, 0.1194, 0.5646]], "s2-47_0008": [["connector_closed", 0.0293, 0.4057, 0.1222, 0.5707]], "s2-47_0018": [["connector_open", 0.082, 0.2856, 0.1825, 0.4627]], "s2-47_0019": [["connector_open", 0.0985, 0.3749, 0.1903, 0.5275]], "s2-47_0020": [["connector_open", 0.0008, 0.302, 0.0751, 0.4857]], "s2-47_0021": [["connector_open", 0.0337, 0.381, 0.1376, 0.5551]], "s2-47_0022": [["connector_open", 0.0474, 0.4058, 0.154, 0.579]], "s2-47_0023": [["connector_open", 0.0374, 0.3766, 0.1469, 0.5536]], "s2-47_0024": [["connector_open", 0.0402, 0.4041, 0.1483, 0.5806]], "s2-47_0025": [["connector_open", 0.0674, 0.4134, 0.1694, 0.5885]], "s2-47_0026": [["connector_open", 0.0634, 0.4093, 0.1726, 0.5905]], "s2-47_0027": [["connector_open", 0.0806, 0.4283, 0.1814, 0.6041]], "s2-47_0028": [["connector_open", 0.0925, 0.4168, 0.1896, 0.5911]], "s2-47_0029": [["connector_open", 0.0858, 0.4487, 0.1776, 0.6186]], "s2-47_0030": [["connector_open", 0.0834, 0.4403, 0.1846, 0.6106]], "s2-47_0031": [["connector_open", 0.1038, 0.4581, 0.2112, 0.6352]], "s2-47_0032": [["connector_open", 0.0842, 0.4435, 0.1841, 0.6106]], "s2-47_0033": [["connector_open", 0.0848, 0.4465, 0.1844, 0.6123]], "s2-47_0034": [["connector_open", 0.1103, 0.4729, 0.201, 0.6322]], "s2-47_0046": [["connector_closed", 0.0585, 0.3716, 0.1563, 0.5501]], "s2-47_0047": [["connector_closed", 0.0573, 0.379, 0.158, 0.5609]], "s2-48_0001": [["connector_closed", 0.1034, 0.3291, 0.1966, 0.4983]], "s2-48_0002": [["connector_closed", 0.1139, 0.3716, 0.2008, 0.5315]], "s2-48_0013": [["connector_open", 0.1524, 0.2866, 0.2467, 0.4449]], "s2-48_0014": [["connector_open", 0.1118, 0.2721, 0.2131, 0.4494]], "s2-48_0015": [["connector_open", 0.1151, 0.2874, 0.2141, 0.4582]], "s2-48_0016": [["connector_open", 0.1008, 0.3201, 0.2035, 0.4887]], "s2-48_0017": [["connector_open", 0.1062, 0.3211, 0.2083, 0.4961]], "s2-48_0018": [["connector_open", 0.0973, 0.2616, 0.2003, 0.4396]], "s2-48_0019": [["connector_open", 0.0852, 0.2692, 0.1929, 0.4403]], "s2-48_0020": [["connector_open", 0.0771, 0.2715, 0.1773, 0.442]], "s2-48_0021": [["connector_open", 0.0773, 0.2999, 0.1869, 0.4746]], "s2-48_0022": [["connector_open", 0.0911, 0.2722, 0.1914, 0.4469]], "s2-48_0023": [["connector_open", 0.102, 0.2799, 0.2093, 0.4556]], "s2-48_0024": [["connector_open", 0.1039, 0.2978, 0.2046, 0.4783]], "s2-48_0025": [["connector_open", 0.11, 0.3175, 0.2131, 0.4792]], "s2-48_0034": [["connector_closed", 0.1025, 0.4065, 0.1907, 0.5701]], "s2-48_0035": [["connector_closed", 0.0682, 0.414, 0.1699, 0.5752]], "s2-48_0036": [["connector_closed", 0.0606, 0.3243, 0.1622, 0.4851]], "s2-48_0037": [["connector_closed", 0.0718, 0.3186, 0.1703, 0.473]], "s2-48_0038": [["connector_closed", 0.0927, 0.3971, 0.2009, 0.571]], "s2-48_0039": [["connector_closed", 0.1001, 0.3914, 0.2008, 0.5679]], "s2-49_0004": [["connector_closed", 0.3088, 0.3613, 0.349, 0.4391]], "s2-49_0005": [["connector_closed", 0.2924, 0.3441, 0.358, 0.459]], "s2-49_0006": [["connector_closed", 0.292, 0.3437, 0.358, 0.4589]], "s2-49_0007": [["connector_closed", 0.2935, 0.3402, 0.3577, 0.456]], "s2-49_0008": [["connector_closed", 0.2942, 0.3422, 0.3578, 0.4566]], "s2-49_0009": [["connector_closed", 0.293, 0.3433, 0.358, 0.4579]], "s2-49_0010": [["connector_closed", 0.2929, 0.3424, 0.3582, 0.4578]], "s2-49_0011": [["connector_closed", 0.2924, 0.3411, 0.359, 0.4591]], "s2-49_0020": [["connector_open", 0.3707, 0.3808, 0.4278, 0.4868]], "s2-49_0021": [["connector_open", 0.2842, 0.3591, 0.3442, 0.4705]], "s2-49_0022": [["connector_open", 0.2004, 0.3503, 0.2716, 0.4743]], "s2-49_0023": [["connector_open", 0.2, 0.3388, 0.2755, 0.4688]], "s2-49_0024": [["connector_open", 0.2224, 0.3874, 0.2986, 0.5219]], "s2-49_0025": [["connector_open", 0.2305, 0.4142, 0.2905, 0.523]], "s2-49_0026": [["connector_open", 0.173, 0.3448, 0.2463, 0.4794]], "s2-49_0044": [["connector_closed", 0.2294, 0.3788, 0.2904, 0.5015]], "s2-49_0045": [["connector_closed", 0.2055, 0.3637, 0.2638, 0.4865]], "s2-49_0046": [["connector_closed", 0.2678, 0.3891, 0.3131, 0.4832]], "s2-49_0047": [["connector_closed", 0.2658, 0.3847, 0.3125, 0.4918]], "s2-49_0048": [["connector_closed", 0.2569, 0.3947, 0.3131, 0.5017]], "s2-49_0049": [["connector_closed", 0.2564, 0.3957, 0.3114, 0.5047]], "s2-50_0001": [["connector_closed", 0.2635, 0.4058, 0.318, 0.5082]], "s2-50_0002": [["connector_closed", 0.2627, 0.4085, 0.3175, 0.5094]], "s2-50_0003": [["connector_closed", 0.2648, 0.4107, 0.3167, 0.5088]], "s2-50_0004": [["connector_closed", 0.2605, 0.5718, 0.329, 0.6861]], "s2-50_0011": [["connector_open", 0.2969, 0.4947, 0.344, 0.5811]], "s2-50_0012": [["connector_open", 0.2692, 0.4632, 0.3187, 0.5637]], "s2-50_0013": [["connector_open", 0.2678, 0.4676, 0.321, 0.5568]], "s2-50_0014": [["connector_open", 0.2666, 0.4671, 0.3202, 0.5643]], "s2-50_0015": [["connector_open", 0.2691, 0.4697, 0.32, 0.5609]], "s2-50_0016": [["connector_open", 0.2353, 0.5025, 0.2896, 0.5983]], "s2-50_0017": [["connector_open", 0.2464, 0.5237, 0.3001, 0.6164]], "s2-50_0018": [["connector_open", 0.2484, 0.5298, 0.2988, 0.6223]], "s2-50_0019": [["connector_open", 0.2477, 0.5189, 0.3006, 0.6115]], "s2-50_0020": [["connector_open", 0.2338, 0.5056, 0.29, 0.5987]], "s2-50_0021": [["connector_open", 0.2432, 0.5276, 0.2979, 0.6199]], "s2-50_0022": [["connector_open", 0.2469, 0.519, 0.2998, 0.6101]], "s2-50_0023": [["connector_open", 0.2469, 0.5238, 0.2999, 0.6157]], "s2-50_0024": [["connector_open", 0.2474, 0.5183, 0.305, 0.6124]], "s2-50_0025": [["connector_open", 0.2537, 0.5291, 0.3039, 0.6198]], "s2-50_0026": [["connector_open", 0.2521, 0.5436, 0.3037, 0.6331]], "s2-50_0027": [["connector_open", 0.2478, 0.5374, 0.3018, 0.6278]], "s2-50_0028": [["connector_open", 0.2568, 0.5244, 0.3053, 0.6109]], "s2-50_0029": [["connector_open", 0.2575, 0.5141, 0.3084, 0.5989]], "s2-50_0030": [["connector_open", 0.2478, 0.5069, 0.3011, 0.6022]], "s2-50_0031": [["connector_open", 0.245, 0.5095, 0.2976, 0.6016]], "s2-50_0032": [["connector_open", 0.2444, 0.5103, 0.2955, 0.5996]], "s2-50_0033": [["connector_open", 0.2461, 0.5187, 0.2966, 0.608]], "s2-50_0034": [["connector_open", 0.215, 0.5144, 0.2687, 0.6055]], "s2-50_0035": [["connector_open", 0.2364, 0.497, 0.2898, 0.5906]], "s2-50_0036": [["connector_open", 0.2342, 0.504, 0.2798, 0.5903]], "s2-50_0040": [["connector_closed", 0.2198, 0.5001, 0.2768, 0.5971]], "s2-50_0041": [["connector_closed", 0.1975, 0.5809, 0.2573, 0.6915]], "s2-50_0042": [["connector_closed", 0.2289, 0.5759, 0.2841, 0.6798]], "s2-50_0043": [["connector_closed", 0.1653, 0.6202, 0.2301, 0.7285]], "s2-50_0044": [["connector_closed", 0.1569, 0.5332, 0.2166, 0.633]], "s2-50_0045": [["connector_closed", 0.2098, 0.4357, 0.2649, 0.5318]], "s2-50_0046": [["connector_closed", 0.2215, 0.4237, 0.2763, 0.516]], "s2-50_0047": [["connector_closed", 0.2254, 0.4323, 0.2845, 0.5347]], "s2-50_0048": [["connector_closed", 0.2249, 0.447, 0.2851, 0.5573]], "s2-50_0049": [["connector_closed", 0.223, 0.4428, 0.2841, 0.5537]], "s2-51_0001": [["connector_closed", 0.2326, 0.7765, 0.2962, 0.8882]], "s2-51_0011": [["connector_open", 0.2535, 0.5021, 0.3083, 0.5948]], "s2-51_0012": [["connector_open", 0.2294, 0.4719, 0.2821, 0.567]], "s2-51_0013": [["connector_open", 0.197, 0.4768, 0.2527, 0.5725]], "s2-51_0014": [["connector_open", 0.1961, 0.4587, 0.2555, 0.5586]], "s2-51_0015": [["connector_open", 0.1916, 0.4554, 0.2522, 0.5582]], "s2-51_0016": [["connector_open", 0.2027, 0.4674, 0.2616, 0.5637]], "s2-51_0017": [["connector_open", 0.2081, 0.4701, 0.2683, 0.5728]], "s2-51_0018": [["connector_open", 0.2275, 0.494, 0.2854, 0.5957]], "s2-51_0019": [["connector_open", 0.236, 0.497, 0.2874, 0.59]], "s2-51_0020": [["connector_open", 0.225, 0.4908, 0.2802, 0.587]], "s2-51_0021": [["connector_open", 0.2319, 0.5027, 0.2883, 0.6008]], "s2-51_0022": [["connector_open", 0.2276, 0.5016, 0.2837, 0.6003]], "s2-51_0023": [["connector_open", 0.2334, 0.5038, 0.2891, 0.5986]], "s2-51_0024": [["connector_open", 0.246, 0.4947, 0.297, 0.5894]], "s2-51_0025": [["connector_open", 0.2445, 0.499, 0.3009, 0.5948]], "s2-51_0026": [["connector_open", 0.2391, 0.4978, 0.291, 0.5887]], "s2-51_0027": [["connector_open", 0.2536, 0.5281, 0.2924, 0.6141]], "s2-51_0028": [["connector_open", 0.2568, 0.5182, 0.3078, 0.6144]], "s2-51_0029": [["connector_open", 0.2559, 0.5159, 0.3056, 0.6028]], "s2-51_0030": [["connector_open", 0.2574, 0.5187, 0.3103, 0.6066]], "s2-51_0034": [["connector_closed", 0.2443, 0.5329, 0.303, 0.6314]], "s2-51_0035": [["connector_closed", 0.1635, 0.5352, 0.2234, 0.6357]], "s2-51_0036": [["connector_closed", 0.1547, 0.5393, 0.2112, 0.6374]], "s2-51_0037": [["connector_closed", 0.1622, 0.4701, 0.2219, 0.5688]], "s2-51_0038": [["connector_closed", 0.1425, 0.4497, 0.2003, 0.5495]], "s2-51_0039": [["connector_closed", 0.1891, 0.4589, 0.2489, 0.5589]], "s2-51_0040": [["connector_closed", 0.1996, 0.4599, 0.2532, 0.5497]], "s2-51_0041": [["connector_closed", 0.2009, 0.4621, 0.2503, 0.5488]], "s2-51_0042": [["connector_closed", 0.1881, 0.446, 0.2445, 0.5437]], "s2-51_0043": [["connector_closed", 0.1993, 0.4461, 0.2482, 0.5346]], "s2-51_0044": [["connector_closed", 0.1939, 0.4372, 0.2502, 0.5425]], "s2-52_0001": [["connector_closed", 0.2083, 0.414, 0.2655, 0.5141]], "s2-52_0002": [["connector_closed", 0.2089, 0.4144, 0.2653, 0.5155]], "s2-52_0003": [["connector_closed", 0.2088, 0.4145, 0.2645, 0.5216]], "s2-52_0004": [["connector_closed", 0.2082, 0.4144, 0.2652, 0.5162]], "s2-52_0005": [["connector_closed", 0.2083, 0.4073, 0.2637, 0.5091]], "s2-52_0006": [["connector_closed", 0.2083, 0.414, 0.2636, 0.5127]], "s2-52_0007": [["connector_closed", 0.2087, 0.4144, 0.2632, 0.5128]], "s2-52_0008": [["connector_closed", 0.2075, 0.4097, 0.2631, 0.5105]], "s2-52_0009": [["connector_closed", 0.2085, 0.4111, 0.2431, 0.4899]], "s2-52_0010": [["connector_closed", 0.206, 0.4085, 0.25, 0.5045]], "s2-52_0012": [["connector_closed", 0.208, 0.414, 0.264, 0.5119]], "s2-52_0013": [["connector_closed", 0.2103, 0.4186, 0.2645, 0.5144]], "s2-52_0014": [["connector_closed", 0.2121, 0.4071, 0.2635, 0.5039]], "s2-52_0015": [["connector_closed", 0.2185, 0.3983, 0.2712, 0.4917]], "s2-52_0016": [["connector_closed", 0.2527, 0.3661, 0.3094, 0.4646]], "s2-52_0017": [["connector_closed", 0.2654, 0.3677, 0.3286, 0.4836]], "s2-52_0018": [["connector_closed", 0.2665, 0.3658, 0.3294, 0.4805]], "s2-52_0019": [["connector_closed", 0.2657, 0.3636, 0.3282, 0.4696]], "s2-52_0028": [["connector_open", 0.1927, 0.3637, 0.2541, 0.4693]], "s2-52_0029": [["connector_open", 0.1547, 0.3811, 0.2271, 0.4963]], "s2-52_0030": [["connector_open", 0.1112, 0.4645, 0.1871, 0.5901]], "s2-52_0031": [["connector_open", 0.1134, 0.529, 0.1914, 0.6519]], "s2-52_0032": [["connector_open", 0.186, 0.5846, 0.2521, 0.6975]], "s2-52_0033": [["connector_open", 0.2055, 0.443, 0.2721, 0.5542]], "s2-52_0034": [["connector_open", 0.2744, 0.3929, 0.3324, 0.4933]], "s2-52_0035": [["connector_open", 0.2701, 0.4056, 0.3267, 0.5044]], "s2-52_0036": [["connector_open", 0.2808, 0.4358, 0.3393, 0.5312]], "s2-52_0037": [["connector_open", 0.2732, 0.4361, 0.3355, 0.5361]], "s2-52_0038": [["connector_open", 0.2686, 0.438, 0.3324, 0.5402]], "s2-52_0039": [["connector_open", 0.2589, 0.4191, 0.3131, 0.509]], "s2-52_0040": [["connector_open", 0.2641, 0.4317, 0.3189, 0.5317]], "s2-52_0041": [["connector_open", 0.2616, 0.4207, 0.3168, 0.513]], "s2-52_0042": [["connector_open", 0.268, 0.4195, 0.3249, 0.5194]], "s2-52_0043": [["connector_open", 0.2699, 0.4048, 0.327, 0.5016]], "s2-52_0044": [["connector_open", 0.2814, 0.4411, 0.3407, 0.5341]], "s2-52_0045": [["connector_open", 0.2819, 0.428, 0.339, 0.5205]], "s2-52_0046": [["connector_open", 0.2785, 0.4372, 0.3346, 0.5309]], "s2-52_0047": [["connector_open", 0.2849, 0.4469, 0.3488, 0.5385]], "s2-52_0048": [["connector_open", 0.3073, 0.4699, 0.3549, 0.56]], "s2-52_0050": [["connector_closed", 0.2759, 0.4269, 0.3291, 0.5264]], "s2-52_0051": [["connector_closed", 0.2909, 0.4475, 0.3473, 0.5402]], "s2-52_0052": [["connector_closed", 0.2779, 0.4176, 0.3313, 0.5118]], "s2-52_0053": [["connector_closed", 0.2033, 0.3834, 0.2547, 0.4881]], "s2-52_0054": [["connector_closed", 0.2256, 0.4676, 0.2906, 0.5897]], "s2-52_0055": [["connector_closed", 0.2033, 0.4736, 0.2705, 0.5902]], "s2-52_0056": [["connector_closed", 0.2422, 0.4108, 0.3021, 0.5208]], "s2-52_0057": [["connector_closed", 0.2732, 0.4298, 0.336, 0.5391]], "s2-52_0058": [["connector_closed", 0.2285, 0.364, 0.2821, 0.4686]], "s2-52_0059": [["connector_closed", 0.2256, 0.3689, 0.2854, 0.4803]], "s2-52_0060": [["connector_closed", 0.2245, 0.3645, 0.284, 0.4747]]}
CONN.update({"s3-17_0005": [["connector_closed", 0.201, 0.2245, 0.2814, 0.2764]], "s3-17_0006": [["connector_closed", 0.2149, 0.2256, 0.2965, 0.2776]], "s3-17_0007": [["connector_closed", 0.2165, 0.225, 0.295, 0.2754]], "s3-17_0008": [["connector_closed", 0.2206, 0.2267, 0.2994, 0.2773]], "s3-17_0009": [["connector_closed", 0.2207, 0.2242, 0.3026, 0.2774]], "s3-17_0010": [["connector_closed", 0.225, 0.2296, 0.3071, 0.284]], "s3-17_0011": [["connector_closed", 0.2206, 0.2337, 0.3002, 0.2872]], "s3-17_0012": [["connector_closed", 0.219, 0.2506, 0.2858, 0.2907]], "s3-17_0013": [["connector_closed", 0.2157, 0.2567, 0.2849, 0.3008]], "s3-17_0014": [["connector_closed", 0.2221, 0.2497, 0.2848, 0.2919]], "s3-17_0015": [["connector_closed", 0.2243, 0.2342, 0.3003, 0.2964]], "s3-17_0016": [["connector_closed", 0.2829, 0.2715, 0.3567, 0.323]], "s3-17_0017": [["connector_closed", 0.4748, 0.3204, 0.5488, 0.3684]], "s3-17_0018": [["connector_closed", 0.4952, 0.3384, 0.5645, 0.3799]], "s3-17_0019": [["connector_closed", 0.4685, 0.3185, 0.5355, 0.3714]], "s3-17_0020": [["connector_closed", 0.4469, 0.3216, 0.5269, 0.3766]], "s3-17_0021": [["connector_closed", 0.4595, 0.3152, 0.5345, 0.3707]], "s3-17_0023": [["connector_closed", 0.4629, 0.3219, 0.5398, 0.3754], ["connector_open", 0.3692, 0.523, 0.4412, 0.5816]], "s3-17_0025": [["connector_closed", 0.4479, 0.3093, 0.5244, 0.3628], ["connector_open", 0.3564, 0.5185, 0.4098, 0.5484]], "s3-17_0034": [["connector_closed", 0.4302, 0.328, 0.5088, 0.3812], ["connector_open", 0.3278, 0.4791, 0.4118, 0.5353]], "s3-17_0035": [["connector_closed", 0.4292, 0.3259, 0.5065, 0.3804], ["connector_open", 0.3351, 0.4875, 0.4046, 0.5298]], "s3-17_0036": [["connector_closed", 0.4254, 0.3249, 0.5011, 0.3774], ["connector_open", 0.3261, 0.4896, 0.3885, 0.5335]], "s3-17_0038": [["connector_closed", 0.4264, 0.3312, 0.5034, 0.3843], ["connector_open", 0.3031, 0.477, 0.3892, 0.5352]], "s3-17_0039": [["connector_closed", 0.4268, 0.3278, 0.5013, 0.3808], ["connector_open", 0.3281, 0.4726, 0.4098, 0.5274]], "s3-17_0040": [["connector_closed", 0.4259, 0.3305, 0.5011, 0.3828], ["connector_open", 0.3253, 0.4864, 0.4051, 0.5299]], "s3-17_0041": [["connector_closed", 0.4254, 0.3283, 0.5005, 0.3812], ["connector_open", 0.3216, 0.4859, 0.4035, 0.5277]], "s3-17_0042": [["connector_closed", 0.4249, 0.329, 0.5026, 0.3823], ["connector_open", 0.2906, 0.4705, 0.3768, 0.5161]], "s3-17_0043": [["connector_closed", 0.4172, 0.3274, 0.4912, 0.3792], ["connector_open", 0.2798, 0.4703, 0.3621, 0.5214]], "s3-17_0049": [["connector_closed", 0.7333, 0.5835, 0.8143, 0.6298], ["connector_open", 0.2449, 0.3181, 0.3192, 0.3752]], "s3-17_0051": [["connector_closed", 0.7652, 0.5923, 0.8433, 0.6438], ["connector_open", 0.2976, 0.3181, 0.3694, 0.3589]], "s3-17_0052": [["connector_closed", 0.7605, 0.5735, 0.8385, 0.6132], ["connector_open", 0.2769, 0.318, 0.3645, 0.3653]], "s3-17_0054": [["connector_closed", 0.7649, 0.5712, 0.8349, 0.606], ["connector_open", 0.2947, 0.325, 0.3738, 0.3742]], "s3-17_0063": [["connector_closed", 0.7684, 0.6149, 0.8499, 0.6652], ["connector_open", 0.2825, 0.378, 0.3691, 0.4285]], "s3-17_0064": [["connector_closed", 0.7779, 0.6235, 0.8628, 0.6738], ["connector_open", 0.3002, 0.3918, 0.3772, 0.439]], "s3-17_0070": [["connector_closed", 0.7831, 0.597, 0.8662, 0.6519], ["connector_open", 0.2991, 0.3734, 0.3655, 0.4226]], "s3-17_0072": [["connector_closed", 0.7982, 0.597, 0.8767, 0.6423], ["connector_open", 0.2956, 0.3908, 0.3678, 0.4332]], "s3-17_0074": [["connector_closed", 0.7827, 0.5938, 0.8636, 0.6381], ["connector_open", 0.3181, 0.3877, 0.3714, 0.4298]], "s3-17_0080": [["connector_closed", 0.8101, 0.5302, 0.8822, 0.579], ["connector_closed", 0.2819, 0.3723, 0.3492, 0.4136]], "s3-17_0082": [["connector_closed", 0.7346, 0.5031, 0.8124, 0.5514], ["connector_closed", 0.1896, 0.3338, 0.2633, 0.3777]], "s3-17_0085": [["connector_closed", 0.7171, 0.5, 0.7968, 0.549], ["connector_closed", 0.1607, 0.3296, 0.2374, 0.3774]], "s3-17_0087": [["connector_closed", 0.7155, 0.5052, 0.7974, 0.5634], ["connector_closed", 0.1619, 0.3378, 0.2432, 0.3882]], "s3-17_0088": [["connector_closed", 0.7457, 0.5032, 0.8268, 0.5615], ["connector_closed", 0.1991, 0.3423, 0.2676, 0.3811]], "s3-17_0090": [["connector_closed", 0.73, 0.5017, 0.8077, 0.5539], ["connector_closed", 0.1781, 0.3381, 0.2554, 0.3805]], "s3-17_0091": [["connector_closed", 0.73, 0.5041, 0.8048, 0.5533], ["connector_closed", 0.1932, 0.339, 0.2521, 0.3799]], "s3-17_0093": [["connector_closed", 0.7296, 0.4978, 0.8105, 0.5514], ["connector_closed", 0.1799, 0.3361, 0.255, 0.3804]], "s3-17_0094": [["connector_closed", 0.7305, 0.4926, 0.8093, 0.5465], ["connector_closed", 0.1823, 0.3267, 0.2566, 0.3718]], "s3-17_0095": [["connector_closed", 0.7308, 0.4912, 0.8099, 0.5436], ["connector_closed", 0.1842, 0.3272, 0.254, 0.3713]], "s3-17_0096": [["connector_closed", 0.7287, 0.4934, 0.8089, 0.5452], ["connector_closed", 0.1733, 0.3254, 0.2506, 0.3708]], "s3-17_0097": [["connector_closed", 0.7254, 0.4938, 0.8022, 0.5494], ["connector_closed", 0.1682, 0.3319, 0.2446, 0.375]], "s3-17_0098": [["connector_closed", 0.7247, 0.4967, 0.803, 0.5517], ["connector_closed", 0.1673, 0.3369, 0.2441, 0.379]], "s3-17_0099": [["connector_closed", 0.7254, 0.4933, 0.7999, 0.5488], ["connector_closed", 0.1631, 0.3259, 0.2375, 0.3759]], "s3-17_0107": [["connector_closed", 0.0374, 0.4776, 0.1213, 0.5365], ["connector_closed", 0.7709, 0.6771, 0.8615, 0.7257]], "s3-17_0108": [["connector_closed", 0.7704, 0.6707, 0.8645, 0.7283], ["connector_closed", 0.0441, 0.4691, 0.1197, 0.5286]], "s3-17_0109": [["connector_closed", 0.0425, 0.4608, 0.1147, 0.5148], ["connector_closed", 0.7706, 0.6551, 0.8628, 0.7109]], "s3-17_0120": [["connector_closed", 0.0545, 0.439, 0.1607, 0.5113], ["connector_closed", 0.7748, 0.6738, 0.8714, 0.747]], "s3-17_0121": [["connector_closed", 0.0626, 0.4304, 0.1656, 0.4998], ["connector_closed", 0.7758, 0.6663, 0.8745, 0.7349]], "s3-17_0130": [["connector_closed", 0.1654, 0.4119, 0.2524, 0.473], ["connector_closed", 0.7623, 0.6427, 0.8538, 0.7006]], "s3-17_0133": [["connector_closed", 0.1589, 0.3985, 0.2475, 0.4625], ["connector_closed", 0.7757, 0.6338, 0.8667, 0.6973]], "s3-17_0134": [["connector_closed", 0.1506, 0.3974, 0.2451, 0.4619], ["connector_closed", 0.7703, 0.6338, 0.8636, 0.696]], "s3-17_0136": [["connector_closed", 0.1504, 0.383, 0.2407, 0.4453], ["connector_closed", 0.7708, 0.6171, 0.8594, 0.6741]], "s3-17_0138": [["connector_closed", 0.1651, 0.3805, 0.2524, 0.4422], ["connector_closed", 0.7688, 0.6147, 0.8522, 0.6755]], "s3-17_0139": [["connector_closed", 0.1656, 0.3673, 0.2554, 0.4289], ["connector_closed", 0.7748, 0.6025, 0.867, 0.6643]], "s3-18_0005": [["connector_closed", 0.4316, 0.2733, 0.5259, 0.3412]], "s3-18_0006": [["connector_closed", 0.4614, 0.2665, 0.5561, 0.328]], "s3-18_0007": [["connector_closed", 0.4505, 0.2625, 0.5469, 0.3272]], "s3-18_0008": [["connector_closed", 0.4353, 0.282, 0.5315, 0.351]], "s3-18_0009": [["connector_closed", 0.4612, 0.2755, 0.559, 0.3492]], "s3-18_0010": [["connector_closed", 0.4329, 0.2381, 0.5362, 0.3091]], "s3-18_0011": [["connector_closed", 0.4405, 0.2234, 0.5464, 0.294]], "s3-18_0012": [["connector_closed", 0.4233, 0.2193, 0.5273, 0.2905]], "s3-18_0013": [["connector_closed", 0.3736, 0.2133, 0.489, 0.2893]], "s3-18_0014": [["connector_closed", 0.3673, 0.2012, 0.4769, 0.2743]], "s3-18_0015": [["connector_closed", 0.3592, 0.2018, 0.4661, 0.2768]], "s3-18_0016": [["connector_closed", 0.3812, 0.1982, 0.4686, 0.2732]], "s3-18_0017": [["connector_closed", 0.372, 0.1997, 0.4669, 0.2727]], "s3-18_0019": [["connector_closed", 0.3696, 0.19, 0.4783, 0.2642]], "s3-18_0043": [["connector_closed", 0.6006, 0.2769, 0.6939, 0.3509], ["connector_open", 0.131, 0.1565, 0.2373, 0.289]], "s3-18_0044": [["connector_closed", 0.6179, 0.2976, 0.6939, 0.3435], ["connector_open", 0.0413, 0.2851, 0.1875, 0.4234]], "s3-18_0045": [["connector_open", 0.0748, 0.3483, 0.1575, 0.4362]], "s3-18_0046": [["connector_open", 0.0329, 0.3476, 0.1503, 0.4517]], "s3-18_0047": [["connector_open", 0.0393, 0.3333, 0.1697, 0.4528]], "s3-18_0048": [["connector_open", 0.0708, 0.3149, 0.1996, 0.4118]], "s3-18_0049": [["connector_open", 0.0402, 0.3074, 0.1624, 0.4055]], "s3-18_0050": [["connector_open", 0.0367, 0.3199, 0.1683, 0.4425]], "s3-18_0051": [["connector_open", 0.0409, 0.3155, 0.1814, 0.4302]], "s3-18_0052": [["connector_open", 0.0666, 0.3051, 0.2071, 0.4286]], "s3-18_0053": [["connector_open", 0.071, 0.2915, 0.1792, 0.4035]], "s3-18_0054": [["connector_open", 0.0625, 0.2731, 0.2047, 0.4222]], "s3-18_0058": [["connector_open", 0.0073, 0.3323, 0.1732, 0.4722]], "s3-18_0061": [["connector_open", 0.0041, 0.3452, 0.1623, 0.471]], "s3-18_0063": [["connector_open", 0.0034, 0.3263, 0.1641, 0.4651]], "s3-18_0064": [["connector_open", 0.0649, 0.3124, 0.1868, 0.4267]], "s3-18_0065": [["connector_open", 0.1855, 0.239, 0.3287, 0.3531]], "s3-18_0066": [["connector_open", 0.1883, 0.2358, 0.3232, 0.3453]], "s3-18_0067": [["connector_open", 0.2126, 0.2391, 0.3459, 0.3452]], "s3-18_0068": [["connector_closed", 0.633, 0.7707, 0.7236, 0.8377], ["connector_open", 0.2168, 0.2438, 0.3502, 0.3485]], "s3-18_0069": [["connector_closed", 0.6429, 0.77, 0.7337, 0.8455], ["connector_open", 0.2705, 0.2524, 0.4039, 0.3545]], "s3-18_0070": [["connector_open", 0.2588, 0.2556, 0.3913, 0.3581]], "s3-18_0071": [["connector_open", 0.1785, 0.2937, 0.3183, 0.4066]], "s3-18_0072": [["connector_open", 0.1352, 0.3004, 0.2783, 0.4093]], "s3-18_0075": [["connector_open", 0.1253, 0.3476, 0.263, 0.4661]], "s3-18_0112": [["connector_closed", 0.6622, 0.2975, 0.7432, 0.3557], ["connector_closed", 0.7949, 0.7143, 0.8974, 0.7698]], "s3-18_0113": [["connector_closed", 0.6519, 0.3, 0.7371, 0.3569], ["connector_closed", 0.7923, 0.7132, 0.8954, 0.7776]], "s3-18_0115": [["connector_closed", 0.5895, 0.2973, 0.671, 0.3544], ["connector_closed", 0.7261, 0.6995, 0.8241, 0.7612]], "s3-18_0116": [["connector_closed", 0.5527, 0.2752, 0.6371, 0.339], ["connector_closed", 0.6779, 0.6758, 0.78, 0.7405]], "s3-18_0117": [["connector_closed", 0.6926, 0.6967, 0.7888, 0.7605], ["connector_closed", 0.554, 0.2954, 0.6449, 0.3592]], "s3-18_0118": [["connector_closed", 0.68, 0.6882, 0.7774, 0.7505], ["connector_closed", 0.5544, 0.2873, 0.6377, 0.3472]], "s3-18_0119": [["connector_closed", 0.5545, 0.2854, 0.6365, 0.3463], ["connector_closed", 0.6849, 0.6805, 0.7868, 0.7501]], "s3-18_0120": [["connector_closed", 0.5642, 0.2523, 0.6493, 0.3183], ["connector_closed", 0.6845, 0.6551, 0.7879, 0.7282]], "s3-18_0121": [["connector_closed", 0.5489, 0.2841, 0.6359, 0.3449], ["connector_closed", 0.6776, 0.6828, 0.7781, 0.7454]], "s3-18_0122": [["connector_closed", 0.668, 0.6946, 0.7653, 0.7528], ["connector_closed", 0.534, 0.2959, 0.6231, 0.3598]], "s3-18_0123": [["connector_closed", 0.6721, 0.6989, 0.7675, 0.7603], ["connector_closed", 0.5371, 0.3043, 0.621, 0.3632]], "s3-p1_0001": [["connector_closed", 0.4702, 0.8225, 0.5701, 0.8896]], "s3-p1_0002": [["connector_closed", 0.5105, 0.9064, 0.6025, 0.9731], ["connector_open", 0.528, 0.1569, 0.6457, 0.2446]], "s3-p1_0003": [["connector_closed", 0.5137, 0.8435, 0.6155, 0.9158], ["connector_closed", 0.664, 0.1988, 0.7749, 0.2777]], "s3-p1_0004": [["connector_closed", 0.3344, 0.7302, 0.4252, 0.7954], ["connector_closed", 0.4646, 0.2004, 0.5614, 0.2671]], "s3-p1_0005": [["connector_closed", 0.6928, 0.291, 0.7746, 0.3551], ["connector_closed", 0.472, 0.7835, 0.5553, 0.8457]], "s3-p1_0006": [["connector_closed", 0.3789, 0.7828, 0.4676, 0.8502], ["connector_closed", 0.6476, 0.2761, 0.7318, 0.3459]], "s3-p1_0007": [["connector_closed", 0.2989, 0.7881, 0.4011, 0.8646], ["connector_closed", 0.6379, 0.2611, 0.7345, 0.336]], "s3-16_0010": [], "s3-16_0012": [], "s3-16_0014": [], "s3-16_0016": [], "s3-16_0018": [], "s3-16_0020": [], "s3-16_0022": [], "s3-17_0001": [], "s3-17_0002": [], "s3-17_0003": [], "s3-17_0004": [], "s3-18_0163": [], "s3-18_0166": [], "s3-18_0170": [], "s3-18_0175": [], "s3-18_0180": [], "s3-18_0206": [], "s3-18_0212": [], "s3-18_0218": []})
EMPTY = set(["s3-16_0010", "s3-16_0012", "s3-16_0014", "s3-16_0016", "s3-16_0018", "s3-16_0020", "s3-16_0022", "s3-17_0001", "s3-17_0002", "s3-17_0003", "s3-17_0004", "s3-18_0163", "s3-18_0166", "s3-18_0170", "s3-18_0175", "s3-18_0180", "s3-18_0206", "s3-18_0212", "s3-18_0218"])   # nothing to label: empty table, phone face down

print(len(CONN), "frames with connector labels,", len(EMPTY), "empty frames")

In [ ]:
# 3b. Turn Grounding DINO's boxes into labels (several wires per frame are allowed)
import numpy as np, cv2

def text_to_class(text):
    t = text.lower()
    if "cutter" in t or "plier" in t: return "cutter"
    if "ruler" in t: return "ruler"
    if "connector" in t or "lever" in t: return "connector?"   # open/closed comes from STATES
    if "wire" in t or "cable" in t:
        for c in ("red", "white", "black"):
            if c in t: return c + "_wire"
        return "wire?"                      # colour decided below
    return None

def red_share(img, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = img[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return 0.0
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    red = ((h <= 10) | (h >= 165)) & (s > 90) & (v > 60)
    return float(red.mean())

def dark_share(img, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = img[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return 0.0
    return float((cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)[..., 2] < 50).mean())

def wire_colour(img, box):
    # Fallback when the text gives no colour: red is easy to measure; dark vs light decides black vs white.
    if red_share(img, box) > 0.03: return "red_wire"
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = cv2.cvtColor(img[max(0, y1):y2, max(0, x1):x2], cv2.COLOR_BGR2HSV)
    v = crop[..., 2].reshape(-1)
    return "black_wire" if (v < 50).mean() > 0.05 else "white_wire"

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0

def inside(a, b):
    # share of box a that lies inside box b
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    area = max(1e-6, (a[2]-a[0]) * (a[3]-a[1]))
    return max(0, x2 - x1) * max(0, y2 - y1) / area

def pick_boxes(img, boxes, scores, texts, state=None, fixed=None):
    H, W = img.shape[:2]
    cand = [[c, 2.0, [x1 * W, y1 * H, x2 * W, y2 * H]] for c, x1, y1, x2, y2 in (fixed or [])]
    for box, score, text in zip(boxes, scores, texts):
        x1, y1, x2, y2 = [float(v) for v in box]
        area = (x2 - x1) * (y2 - y1) / (W * H)
        if area > 0.6 or area < 0.0003:
            continue
        cls = text_to_class(text)
        if cls is None:
            continue
        if cls == "wire?":
            cls = wire_colour(img, box)
        elif cls == "connector?" and fixed is not None:
            continue                        # connectors come from CONN
        elif cls == "connector?":
            if globals().get("ARRIVED_LEFT_OF") and (x1 + x2) / 2 < ARRIVED_LEFT_OF * W:
                cls = "connector_closed"    # the arrived unit's connector: closed at the last station
            elif state in ("open", "closed"):
                cls = "connector_" + state
            else:
                continue
        elif cls.endswith("_wire"):
            r = red_share(img, box)
            if cls == "red_wire" and r < 0.005 and score < 0.5: cls = wire_colour(img, box)
            elif cls != "red_wire" and r > 0.08: cls = "red_wire"     # clearly red: fix the text
        if cls not in CLASSES:
            continue
        bw, bh = x2 - x1, y2 - y1
        if cls == "ruler" and max(bw, bh) < 3 * min(bw, bh):
            continue                        # a ruler is long and thin; this is the whole cutting board
        if cls == "red_wire" and red_share(img, box) < 0.01:
            continue                        # nothing red in it
        if cls == "black_wire" and dark_share(img, box) < 0.03:
            continue                        # nothing black in it (e.g. the board's metal rail)
        cand.append([cls, float(score), [x1, y1, x2, y2]])
    cand.sort(key=lambda c: -c[1])
    keep = []
    cutters = [c[2] for c in cand if c[0] == "cutter"]
    conns = [c[2] for c in cand if c[0].startswith("connector")]
    for c in cand:                          # one box per object: drop a box that mostly repeats a better one
        if c[1] == 2.0:                     # a hand-checked box (CONN): always kept
            keep.append(c); continue
        if any(iou(c[2], k[2]) > 0.6 for k in keep):
            continue
        if c[0].endswith("_wire") and any(inside(c[2], k) > 0.5 for k in cutters):
            continue                        # the cutter's red handles are not a red wire
        if c[0].endswith("_wire") and any(inside(c[2], k) > 0.7 for k in conns):
            continue                        # the connector's orange levers are not a red wire
        if c[0].startswith("connector") and sum(k[0].startswith("connector") for k in keep) >= globals().get("MAX_CONNECTORS", 1):
            continue                        # no more connectors than this station can have
        if c[0] in ONE_PER_IMAGE and any(k[0] == c[0] for k in keep):
            continue
        keep.append(c)
    return [(c[0], c[2]) for c in keep]

def to_yolo_lines(objs, W, H):
    out = []
    for cls, (x1, y1, x2, y2) in objs:
        x1, x2 = max(0, x1), min(W, x2); y1, y2 = max(0, y1), min(H, y2)
        out.append(f"{CLASSES.index(cls)} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    return out

In [ ]:
# 3c. Auto-label every frame (a few minutes on a T4)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(DINO_MODEL)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL).to(device).eval()

def detect(pil):
    inputs = processor(images=pil, text=PROMPT, return_tensors="pt").to(device)
    with torch.no_grad():
        out = dino(**inputs)
    kw = dict(text_threshold=TEXT_THRESHOLD, target_sizes=[pil.size[::-1]])
    try:
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, threshold=BOX_THRESHOLD, **kw)[0]
    except TypeError:
        r = processor.post_process_grounded_object_detection(out, inputs.input_ids, box_threshold=BOX_THRESHOLD, **kw)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy(), list(r.get("text_labels", r.get("labels")))

LABELS = WORK / "labels_all"
LABELS.mkdir(exist_ok=True)
SKIPPED = []                                 # frames whose connector state is "skip"
has_states = "STATES" in globals()
counts = {c: 0 for c in CLASSES}
for f in tqdm(FRAMES):
    pil = Image.open(f).convert("RGB")
    img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    if Path(f).stem in globals().get("EMPTY", set()):
        (LABELS / (Path(f).stem + ".txt")).write_text(""); continue   # no boxes at all
    state = frame_state(Path(f).stem) if has_states else None
    if state == "skip" or ("CONN" in globals() and Path(f).stem not in CONN):
        SKIPPED.append(Path(f).stem); continue
    objs = pick_boxes(img, *detect(pil), state=state, fixed=globals().get("CONN", {}).get(Path(f).stem))
    for c, _ in objs: counts[c] += 1
    (LABELS / (Path(f).stem + ".txt")).write_text("\n".join(to_yolo_lines(objs, *pil.size)))
print("Boxes per class:", counts)
if SKIPPED: print(len(SKIPPED), "frames skipped (levers changing):", SKIPPED[0], "...", SKIPPED[-1])

## 4. Review every frame

The pages below show **all** frames, 24 at a time, with their boxes:
**red** = red_wire, **white** = white_wire, **black (yellow outline)** = black_wire, **orange** = cutter, **blue** = ruler,
**green** = connector_open, **purple** = connector_closed. Skipped frames show "SKIP" and are left out anyway.

Write down the names of frames that are clearly wrong (a missing wire, a wrong colour, a box on a hand),
put them in `BAD` in cell 4b and run it. Those frames are left out of training. A few small misses are fine.
If most frames are wrong, change `BOX_THRESHOLD` in 3a (try 0.25 or 0.35) and run 3c again.

In [ ]:
# 4a. Show all labeled frames, one page at a time. Change PAGE and run again (PAGE = 0, 1, 2, ...)
import matplotlib.pyplot as plt
PAGE = 0
COLORS = {"red_wire": (230, 40, 40), "white_wire": (255, 255, 255), "black_wire": (255, 220, 0),
          "cutter": (255, 140, 0), "ruler": (0, 150, 255),
          "connector_open": (0, 200, 0), "connector_closed": (170, 0, 255)}

def draw(f):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    lab = LABELS / (Path(f).stem + ".txt")
    for line in (lab.read_text().splitlines() if lab.exists() else []):
        c, x, y, w, h = line.split(); name = CLASSES[int(c)]
        x, y, w, h = float(x)*W, float(y)*H, float(w)*W, float(h)*H
        cv2.rectangle(img, (int(x-w/2), int(y-h/2)), (int(x+w/2), int(y+h/2)), COLORS[name], max(2, W//250))
    return img

SHOW = [f for f in FRAMES if Path(f).stem not in set(globals().get("SKIPPED", []))]   # skipped frames are left out anyway
pages = (len(SHOW) + 23) // 24
chunk = SHOW[PAGE*24:(PAGE+1)*24]
print(f"Page {PAGE} of {pages - 1}")
fig, axes = plt.subplots(4, 6, figsize=(24, 18))
for ax in axes.flat: ax.axis("off")
for ax, f in zip(axes.flat, chunk):
    skip = Path(f).stem in set(globals().get("SKIPPED", []))
    ax.imshow(draw(f)); ax.set_title(Path(f).stem + ("  SKIP" if skip else ""), fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 4b. Frames to leave out (names as shown above), e.g. BAD = ["sample1-S1_0042", "sample2-S1_0107"]
BAD = []
USE = [f for f in FRAMES if Path(f).stem not in set(BAD) | set(globals().get("SKIPPED", []))]
print(len(USE), "frames kept,", len(FRAMES) - len(USE), "left out")

In [ ]:
# 5. Build the dataset. Split by video, never by frame: VAL_VIDEOS (or with 3+ videos, every 7th) are the check set;
#    with fewer, every 4th 2-second block of each video is (continuous stretches, so every step of the video
#    is in both sets, but neighbouring near-identical frames never end up on both sides).
import yaml, random
DS = WORK / "dataset"
shutil.rmtree(DS, ignore_errors=True)
by_video = {}
for f in USE: by_video.setdefault(Path(f).stem.rsplit("_", 1)[0], []).append(f)
train, val = [], []
if VAL_VIDEOS or len(by_video) >= 3:
    hold = set(VAL_VIDEOS) or set(sorted(by_video)[3::7]) or {sorted(by_video)[-1]}
    for v, fs in by_video.items(): (val if v in hold else train).extend(fs)
    print("Check set = videos", sorted(hold))
else:
    for v, fs in by_video.items():
        for f in sorted(fs):
            block = int(Path(f).stem.rsplit("_", 1)[1]) // (2 * FPS)
            (val if block % 4 == 3 else train).append(f)
    print("Check set = every 4th 2-second block of each video")
for split, fs in (("train", train), ("val", val)):
    (DS / "images" / split).mkdir(parents=True); (DS / "labels" / split).mkdir(parents=True)
    for f in fs:
        shutil.copy(f, DS / "images" / split)
        lab = LABELS / (Path(f).stem + ".txt")
        (DS / "labels" / split / lab.name).write_text(lab.read_text() if lab.exists() else "")
(DS / "data.yaml").write_text(yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val",
                                             "names": {i: n for i, n in enumerate(CLASSES)}}))
print("train:", len(train), " check:", len(val))

In [ ]:
# 6. Train (10-20 min on a T4)
from ultralytics import YOLO
try:
    model = YOLO("yolo26n.pt")
except Exception:
    model = YOLO("yolo11n.pt")
model.train(
    data=str(DS / "data.yaml"), epochs=80, patience=25, imgsz=640, batch=16,
    project=str(WORK / "runs"), name=STATION, exist_ok=True,
    # Overhead camera: any rotation or flip is realistic; strong brightness and colour changes = robust to lighting.
    # hsv_h stays small so red, white and black keep their colour.
    degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0.01, hsv_s=0.5, hsv_v=0.5, scale=0.4,
)
BEST = WORK / "runs" / STATION / "weights" / "best.pt"
print("Saved:", BEST)

In [ ]:
# 7. How good is it? mAP50 near 1.0 = very good, above 0.8 = fine to start.
best = YOLO(str(BEST))
m = best.val(data=str(DS / "data.yaml"), verbose=False)
print(f"mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}")
for i, n in enumerate(CLASSES):
    ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist())).get(i)
    print(f"  {n:11s} " + (f"mAP50 = {ap:.3f}" if ap is not None else "not in the check set"))
show = sorted(glob.glob(str(DS / "images" / "val" / "*.jpg")))[::max(1, len(val) // 6)][:6]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 6))
for ax, f in zip(np.atleast_1d(axes), show):
    ax.imshow(best.predict(f, conf=0.4, verbose=False)[0].plot()[:, :, ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 7b. Score the model /line uses now (s3-connector-new.onnx from app/models/) on the same check set.
#     Swap only if 7 beats it, especially on connector_open and on the live session s3-17.
from google.colab import files
if not os.path.exists("s3-connector-new.onnx"):
    print("Choose app/models/s3-connector-new.onnx ...")
    files.upload()
old = YOLO("s3-connector-new.onnx", task="detect")
mo = old.val(data=str(DS / "data.yaml"), imgsz=640, verbose=False)
print(f"CURRENT model  mAP50: {mo.box.map50:.3f}   (new model: {m.box.map50:.3f})")
olda = dict(zip(mo.box.ap_class_index.tolist(), mo.box.ap50.tolist()))
newa = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist()))
for i, n in enumerate(CLASSES):
    print(f"  {n:17s} current {olda.get(i, 0):.3f}   new {newa.get(i, 0):.3f}")


In [ ]:
# 8. Export to ONNX for /line (opset 12, 640 px, raw head) and download it, plus best.pt and the labeled data
onnx_path = best.export(format="onnx", opset=12, imgsz=640, simplify=False)
shutil.copy(onnx_path, f"/content/{EXPORT_NAME}")
shutil.copy(BEST, f"/content/{STATION}-best.pt")
shutil.make_archive(f"/content/{STATION}-dataset", "zip", DS)
print("Classes in order:", CLASSES)
from google.colab import files
files.download(f"/content/{EXPORT_NAME}")
files.download(f"/content/{STATION}-best.pt")